In [ ]:
import os
import re
import pickle
from collections import defaultdict
from pathlib import Path
import random

from typing import Dict, List, Tuple, Union, Optional

import seaborn as sns
import matplotlib.pyplot as plt
from matplotlib import font_manager
from matplotlib.lines import Line2D
from matplotlib.colors import LinearSegmentedColormap
from PIL import Image, ImageDraw, ImageFont

import numpy as np
import pandas as pd
from scipy.stats import spearmanr, norm
from sklearn.cluster import KMeans
from sklearn.metrics import precision_score, recall_score, f1_score
from sklearn.decomposition import PCA
from sklearn.neighbors import NearestNeighbors
from sklearn.model_selection import train_test_split
from sklearn.linear_model import Lasso
from sklearn.preprocessing import StandardScaler
import torch
from uncertainty_toolbox.metrics_calibration import get_proportion_lists

from pymatgen.core import Composition
from pymatgen.core.composition import Composition
from smact.metallicity import metallicity_score

from evidential_regression.data import data_utils
from evidential_regression.config.config import CONFIG
from evidential_regression.data.data_utils import create_dataloaders
from evidential_regression.models.network import FFNN
from evidential_regression.models.init_utils import init_weights
from evidential_regression.models.early_stopping import EarlyStopper
from evidential_regression.training.train_eval import (
    train_and_val_proc,
    test_proc,
    val_proc,
)
from evidential_regression.training.losses import nig_nll, evidential_regression
from evidential_regression.training import metrics
from evidential_regression.utils.seed import set_seed

from periodic_table_plot import generate_periodic_table_heatmap

# 1- Preprocessing 

In [ ]:
valid_elements = {
    "H",
    "He",
    "Li",
    "Be",
    "B",
    "C",
    "N",
    "O",
    "F",
    "Ne",
    "Na",
    "Mg",
    "Al",
    "Si",
    "P",
    "S",
    "Cl",
    "Ar",
    "K",
    "Ca",
    "Sc",
    "Ti",
    "V",
    "Cr",
    "Mn",
    "Fe",
    "Co",
    "Ni",
    "Cu",
    "Zn",
    "Ga",
    "Ge",
    "As",
    "Se",
    "Br",
    "Kr",
    "Rb",
    "Sr",
    "Y",
    "Zr",
    "Nb",
    "Mo",
    "Tc",
    "Ru",
    "Rh",
    "Pd",
    "Ag",
    "Cd",
    "In",
    "Sn",
    "Sb",
    "Te",
    "I",
    "Xe",
    "Cs",
    "Ba",
    "La",
    "Ce",
    "Pr",
    "Nd",
    "Pm",
    "Sm",
    "Eu",
    "Gd",
    "Tb",
    "Dy",
    "Ho",
    "Er",
    "Tm",
    "Yb",
    "Lu",
    "Hf",
    "Ta",
    "W",
    "Re",
    "Os",
    "Ir",
    "Pt",
    "Au",
    "Hg",
    "Tl",
    "Pb",
    "Bi",
    "Po",
    "At",
    "Rn",
    "Fr",
    "Ra",
    "Ac",
    "Th",
    "Pa",
    "U",
    "Np",
    "Pu",
    "Am",
    "Cm",
    "Bk",
    "Cf",
    "Es",
    "Fm",
    "Md",
    "No",
    "Lr",
    "Rf",
    "Db",
    "Sg",
    "Bh",
    "Hs",
    "Mt",
    "Ds",
    "Rg",
    "Cn",
    "Nh",
    "Fl",
    "Mc",
    "Lv",
    "Ts",
    "Og",
}


def chemical_formula_validity(formula: str) -> bool:
    """
    Check if the chemical formula is valid or not.

    Parameters
    ----------
    formula : str
        Chemical formula string, e.g., Ti0.99Nb0.01NiSn or (PbTe)0.05(Ag2Te)0.95.

    Returns
    -------
    bool
        Return true or false for validity.
    """
    if not isinstance(formula, str):
        return False

    # Regex for element: e.g., Pb, Te, Ag2, Se0.92, Br0.08
    element = r"[A-Z][a-z]?(?:\d+(?:\.\d+)?|\.\d+)?"

    # Regex for group: e.g., (PbTe)0.05, (Se0.92Br0.08)3
    group = r"\((?:{elem})+\)(?:\d+(?:\.\d+)?|\.\d+)?".format(elem=element)

    # Combine: allow a formula to be a sequence of elements or groups
    pattern = f"^(?:{element}|{group})+$"

    if not re.fullmatch(pattern, formula):
        return False

    # Extract all element symbols without their numbers or groups
    elements = re.findall(r"[A-Z][a-z]?", formula)
    if not all(el in valid_elements for el in elements):
        return False

    try:
        Composition(formula)
        return True
    except Exception:
        return False


def safe_metallicity_score(comp: str) -> Optional[float]:
    """
    Compute the metallicity score for a given composition safely.

    Parameters
    ----------
    comp : str
        Chemical composition string.

    Returns
    -------
    float or None
        The metallicity score for the composition if successfully computed;
        None if a ZeroDivisionError occurs during computation.

    Notes
    -----
    This function wraps `metallicity_score` and catches a potential
    `ZeroDivisionError`. If such an error occurs, it prints the problematic
    composition and returns None.
    """
    try:
        return metallicity_score(comp)
    except ZeroDivisionError:
        print(comp)
        return None


def to_hill_notation_string(formula_str: str) -> str:
    """Converts a chemical formula string into Hill notation.

    This function parses a potentially complex chemical formula and returns a
    standardized string representation following the Hill system. It correctly
    handles decimal stoichiometries, parenthetical groups, and solid solutions.

    Parameters
    ----------
    formula_str : str
        The chemical formula string to be parsed. This can be in a complex
        format, e.g., '(PbTe)0.05(Ag2Te)0.95'.

    Returns
    -------
    str
        A standardized string representation of the chemical composition in
        Hill notation (C, then H, then other elements alphabetically).
        For example, 'Ag1.9Pb0.05Te1'.

    """
    comp = Composition(formula_str)
    el_amt_dict = comp.get_el_amt_dict()

    sorted_elements = []

    if "C" in el_amt_dict:
        sorted_elements.append("C")

    if "H" in el_amt_dict:
        sorted_elements.append("H")

    other_elements = sorted([el for el in el_amt_dict if el not in ["C", "H"]])
    sorted_elements.extend(other_elements)

    hill_string_parts = []
    for el in sorted_elements:
        amount = el_amt_dict[el]
        if float(amount).is_integer():
            amount_str = str(int(amount))
        else:
            amount_str = str(round(amount, 4))

        hill_string_parts.append(f"{el}{amount_str}")

    return "".join(hill_string_parts)


CATION_ELEMENTS = {
    "Li",
    "Na",
    "K",
    "Rb",
    "Cs",
    "Fr",
    "Be",
    "Mg",
    "Ca",
    "Sr",
    "Ba",
    "Ra",
    "Sc",
    "Y",
    "Ti",
    "Zr",
    "Hf",
    "V",
    "Nb",
    "Ta",
    "Cr",
    "Mo",
    "W",
    "Mn",
    "Tc",
    "Re",
    "Fe",
    "Ru",
    "Os",
    "Co",
    "Rh",
    "Ir",
    "Ni",
    "Pd",
    "Pt",
    "Cu",
    "Ag",
    "Au",
    "Zn",
    "Cd",
    "Hg",
    "Al",
    "Ga",
    "In",
    "Tl",
    "Pb",
    "Sn",
    "Bi",
}


def to_iupac_notation_string(formula_str: str) -> str:
    """
    Converts a chemical formula into IUPAC-style cation–anion order.

    Parameters
    ----------
    formula_str : str
        The chemical formula to parse.

    Returns
    -------
    str
        A standardized string with cations listed first, then anions, each
        followed by its stoichiometric amount.
    """
    comp = Composition(formula_str)
    el_amt_dict = comp.get_el_amt_dict()

    cations = []
    anions = []

    for el in el_amt_dict:
        if el in CATION_ELEMENTS:
            cations.append(el)
        else:
            anions.append(el)

    cations.sort()
    anions.sort()

    ordered_elements = cations + anions

    parts = []
    for el in ordered_elements:
        amt = el_amt_dict[el]
        if float(amt).is_integer():
            amt_str = str(int(amt))
        else:
            amt_str = str(round(amt, 4))
        parts.append(f"{el}{amt_str}")

    return "".join(parts)


def to_kelvin(temp_str: Union[str, float, int]) -> Optional[float]:
    """
    Convert a temperature representation into Kelvin.

    Parameters
    ----------
    temp_str : str, float, or int
        Temperature value expressed as a string (e.g., "300 K", "25 °C",
        "room temperature") or as a numeric type. Non-string values are
        ignored and return None.

    Returns
    -------
    float or None
        Temperature in Kelvin as a float if conversion is successful.
        Returns None if the input cannot be interpreted as a single
        temperature value.
    """
    if not isinstance(temp_str, str):
        return None

    s = temp_str.strip().lower()

    # --- Handle room/ambient synonyms ---
    if re.search(r"\b(room temperature|ambient|r\.?t\.?)\b", s):
        return 298.0

    # --- Match Kelvin values ---
    match_k = re.fullmatch(r"(\d+(\.\d+)?)\s*k", s, re.IGNORECASE)
    if match_k:
        return float(match_k.group(1))

    # --- Match Celsius values ---
    match_c = re.fullmatch(r"(\d+(\.\d+)?)\s*°?c", s, re.IGNORECASE)
    if match_c:
        celsius = float(match_c.group(1))
        return celsius + 273.15

    # --- Otherwise ---
    return None


def preprocess(csv_file: str) -> pd.DataFrame:
    """Preprocess a material dataset throught several steps:
    check the validity of chemical formula, applying IUPAC notation,
    discarding entries with NaN values, metal material,
    and outlier data instances with IQR method.

    Parameters
    ----------
    csv_file : str
        A path to a csv file (e.g., estm.csv).

    Returns
    -------
    pd.DataFrame
        A preprocessed dataset in pandas DataFrame.
    """
    df = pd.read_csv(csv_file)
    initial_num = df.shape[0]
    df = df.loc[:, ["composition", "k", "T"]]

    df = df[df.loc[:, "composition"].map(chemical_formula_validity)]

    df["iupac_notation"] = df.loc[:, "composition"].apply(to_iupac_notation_string)

    df.dropna(inplace=True)
    df.drop_duplicates(inplace=True)

    scores = df["composition"].map(safe_metallicity_score)
    non_metal_df = df[scores.notnull() & (scores < 0.7)].reset_index(drop=True)

    q1 = non_metal_df["k"].quantile(0.25)
    q3 = non_metal_df["k"].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    non_metal_df["outlier"] = (non_metal_df["k"] < lower_bound) | (
        non_metal_df["k"] > upper_bound
    )
    clean_non_metal_df = non_metal_df[~non_metal_df["outlier"]]
    clean_non_metal_df = clean_non_metal_df.drop(columns=["outlier", "composition"])

    q1 = clean_non_metal_df["T"].quantile(0.25)
    q3 = clean_non_metal_df["T"].quantile(0.75)
    iqr = q3 - q1
    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr
    clean_non_metal_df["outlier"] = (clean_non_metal_df["T"] < lower_bound) | (
        clean_non_metal_df["T"] > upper_bound
    )
    clean_non_metal_df = clean_non_metal_df[~clean_non_metal_df["outlier"]]
    clean_non_metal_df = clean_non_metal_df.drop(columns=["outlier"])

    clean_non_metal_df["k"] = clean_non_metal_df["k"].round(4)
    clean_non_metal_df["T"] = clean_non_metal_df["T"].astype(float)

    clean_non_metal_df.drop(
        clean_non_metal_df[
            (clean_non_metal_df["T"] < 0) | (clean_non_metal_df["k"] <= 0)
        ].index,
        axis=0,
        inplace=True,
    )

    clean_non_metal_df.to_csv(
        f"../dataset/clean_{csv_file.split('/')[-1][:-4]}.csv", index=False
    )

    print(
        f"{csv_file.split('/')[-1][:-4].upper()}: Cleaning procedure reduce number of data from {initial_num} to {clean_non_metal_df.shape[0]}."
    )

    return clean_non_metal_df


def drop_identical_entries(
    df1: pd.DataFrame,
    df2: pd.DataFrame,
    name: str,
    subset: List = ["k", "iupac_notation", "T"],
) -> pd.DataFrame:
    """Drop identical entries respect to ESTM dataframe (df1) from df2.

    Parameters
    ----------
    df1 : pd.DataFrame
        Preprocessed ESTM dataframe.
    df2 : pd.DataFrame
        Another preprocessed dataframe.
    name: str
        Name of another preprocessed dataframe.
    subset: List
        Name of columns to find identical entries [k, hill_notation, T].

    Returns
    -------
    pd.DataFrame
        Filtered dataset.
    """
    common_df = pd.merge(df1, df2, on=subset, how="inner")
    filtered_df2 = df2[
        ~df2.set_index(subset).index.isin(common_df.set_index(subset).index)
    ]
    filtered_df2.reset_index(drop=True)

    filtered_df2.to_csv(f"../dataset/filtered_{name}.csv", index=False)

    print(
        f"{name.upper()}: The size of dataset reduces from {df2.shape[0]} to {filtered_df2.shape[0]}."
    )

    return filtered_df2


def get_formula_dict(formula: str) -> Dict[str, float]:
    """
    Parse a chemical formula into a dict of element amounts using pymatgen.

    Parameters
    ----------
    formula : str
        Chemical formula string (e.g., "Bi1.2Co0.1Ti1.9S5.2").

    Returns
    -------
    Dict[str, float]
        Element symbols mapped to their quantities.
    """
    comp = Composition(formula)
    return comp.get_el_amt_dict()


def get_feature(formula: str, temp: float, kl: float) -> np.array:
    """Generate feature vector comprising elemental vector, descriptors, and temperature.

    Parameters
    ----------
    formula : str
        String of a chemical formula.
    temp : float
        A temperature.
    kl : float
        Lattice thermal conductivity value.

    Returns
    -------
    np.array
        Generated features.
    """

    comp = get_formula_dict(formula)

    all_elems = oliynyk_df.index.tolist()
    stoich_vec = np.array([comp.get(el, 0.0) for el in all_elems])

    props = oliynyk_df.columns
    weighted = {}
    fractions = np.array([comp.get(el, 0.0) for el in all_elems])

    for prop in props:
        vals = oliynyk_df[prop].values

        w = fractions * vals
        mean = np.sum(w) / fractions.sum()
        weighted[f"{prop}_mean"] = mean

        variance = np.sum(fractions * (vals - mean) ** 2) / fractions.sum()
        weighted[f"{prop}_variance"] = variance

        weighted[f"{prop}_std"] = np.sqrt(variance)

        mask = vals > 0
        vals_masked = vals[mask]
        fractions_masked = fractions[mask]

        harmonic_mean = np.sum(fractions_masked) / np.sum(
            fractions_masked / vals_masked
        )
        weighted[f"{prop}_harmonic_mean"] = harmonic_mean

        mask_frac = fractions > 0
        weighted[f"{prop}_max"] = np.max(vals[mask_frac])
        weighted[f"{prop}_min"] = np.min(vals[mask_frac])

    feature_vector = np.concatenate(
        [[formula], stoich_vec, np.array(list(weighted.values())), [temp], [kl]]
    )

    return feature_vector


def get_all_feature(df: pd.DataFrame) -> np.array:
    """Generate feature vector for a series of materials.

    Parameters
    ----------
    df : pd.DataFrame
        A dataframe containing kL, T, and chemical formula

    Returns
    -------
    np.array
        Generated feature vectors for all materials.
    """

    fv_list = []
    for row in df.itertuples(index=True):
        fv = get_feature(row.iupac_notation, row.T, row.k)
        fv_list.append(fv)
    fvs = np.vstack(fv_list)

    with open("feat_names.txt", "r") as file:
        lines = file.readlines()
        feat_names = [line.strip() for line in lines]

    fvs_df = pd.DataFrame(fvs, columns=feat_names)

    fvs_df.iloc[:, 1:] = fvs_df.iloc[:, 1:].astype(float)

    return fvs_df

In [ ]:
df_list = []
for i in ["ucsb", "estm", "cher", "citrine", "llm_dataset"]:
    df = pd.read_csv(f"../dataset/{i}.csv")
    df["name"] = i
    df_list.append(df)

pd.concat(df_list).to_csv("../dataset/total_dataset.csv", index=False)

In [ ]:
total_df = preprocess("../dataset/total_dataset.csv")
total_df = total_df.drop_duplicates(["T", "iupac_notation"], keep="first")
total_df.to_csv("../dataset/clean_total_dataset.csv", index=False)

starry_df = preprocess("../dataset/starrydataset.csv")

filtered_starry_df = drop_identical_entries(
    total_df, starry_df, "starrydataset", ["k", "iupac_notation", "T"]
)

new_filtered_starry_df = filtered_starry_df.drop_duplicates(
    subset=["T", "iupac_notation"]
)

new_filtered_starry_df.to_csv(
    "../dataset/unique_filtered_starrydataset.csv", index=False
)

# 2- Generate Feature Vector

In [ ]:
oliynyk_df = pd.read_excel("../dataset/oliynyk-elemental-property-list.xlsx")
oliynyk_df.set_index("Symbol", inplace=True)

oliynyk_df = oliynyk_df.loc[
    :,
    [
        "Pauling \nEN",
        "Ghosh\nEN",
        "Ionization\nenergy (eV)",
        "Atomic \nweight",
        "no. of \nvalence \nelectrons",
        "Covalent\nradius",
        "Density, \ng/mL",
        "Specific heat,\nJ/g K",
        "polarizability,\n A^3",
    ],
]
oliynyk_df.fillna(0.0, inplace=True)

In [ ]:
total_df = pd.read_csv("../dataset/clean_total_dataset.csv")
total_fv_df = get_all_feature(total_df)
total_fv_df["name"] = total_df["name"]
total_fv_df.to_parquet("../feature_vectors/total_dataset.parquet", index=False)

In [ ]:
unique_filtered_starry_df = pd.read_csv("../dataset/unique_filtered_starrydataset.csv")
unique_filtered_starry_df = get_all_feature(unique_filtered_starry_df)
unique_filtered_starry_df["name"] = "starry"
unique_filtered_starry_df.to_parquet(
    "../feature_vectors/unique_starrydataset.parquet", index=False
)

# 3- Feature Engineerning

In [ ]:
def feature_discard(
    df: pd.DataFrame, var_thershold: float = 0.01, corr_threshold: float = 0.95
) -> pd.DataFrame:
    """Remove static, quasi-static, and correlated features, then applies LASSO to select subset of features.
    Apply only on OLED features.

    Parameters
    ----------
    df : pd.DataFrame
        Provided dataframe of generated features.
    var_thershold : float, optional
        Threshold of variance, by default 0.01
    corr_threshold : float, optional
        Threshold of correlation, by default 0.95

    Returns
    -------
    pd.DataFrame
        Filtered dataframe.
    """
    df = df.loc[
        :, [True] * 86 + list(df.iloc[:, 86:-1].var(axis=0) > var_thershold) + [True]
    ]

    corr_matrix = df.iloc[:, 86:-1].corr().abs()
    upper_tri = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
    to_drop = [
        column
        for column in upper_tri.columns
        if any(upper_tri[column] >= corr_threshold)
    ]
    df = df.drop(columns=to_drop)

    scaler = StandardScaler()
    X = scaler.fit_transform(df.iloc[:, 86:-1])
    y = df.iloc[:, -1]
    reg = Lasso(alpha=0.01)
    reg.fit(X, y)

    df = df.loc[:, [True] * 86 + list(reg.coef_ != 0) + [True]]

    return df

In [ ]:
starry_fv_df = pd.read_parquet("./feature_vectors/starry_fv.parquet")
filtered_starry_fv_df = feature_discard(starry_fv_df)

# 4- Model

## 4-1- Train/Validation/Test

In [ ]:
df_all = pd.read_parquet("../feature_vectors/total_dataset.parquet")
df_telab = pd.read_parquet("../feature_vectors/unique_telab_dataset.parquet")

In [ ]:
def create_strata(df, n_bins=5):
    """Creates a combined stratification column based on kL and T quantiles."""

    df[f"kL_bins"] = pd.qcut(df["kL"], q=n_bins, labels=False, duplicates="drop")
    df[f"T_bins"] = pd.qcut(df["T"], q=n_bins, labels=False, duplicates="drop")

    df["Strata"] = df["T_bins"].astype(str) + "_" + df["kL_bins"].astype(str)
    return df

In [ ]:
def run_experiment(
    df_all: pd.DataFrame,
    df_ext: pd.DataFrame,
    runs: int,
    output_dir: Union[str, Path],
) -> None:
    """
    Run experiments across multiple random seeds and model instances.

    Parameters
    ----------
    df_all : pandas.DataFrame
        Input dataset containing the target column `col_name`.
    df_ext: pandas.DataFrame
        External test set.
    runs : int
        Number of independent experiment runs.
    output_dir : str or pathlib.Path
        Directory path for saving outputs, model checkpoints, and split files.
    """
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    output_dir = Path(output_dir)

    df_all = create_strata(df_all)

    for run_idx in range(1, runs + 1):
        print(f"\nRun: {run_idx}")
        random_seed = 42 + run_idx

        run_dir = output_dir / f"run_{run_idx}"
        run_dir.mkdir(parents=True, exist_ok=True)

        df_train, df_remain = train_test_split(
            df_all,
            test_size=800,
            stratify=df_all["Strata"],
            random_state=random_seed,
        )

        df_test, df_val = train_test_split(
            df_remain,
            test_size=400,
            stratify=df_remain["Strata"],
            random_state=random_seed,
        )

        train_set = (
            df_train.drop(columns=["T_bins", "kL_bins", "Strata"])
            .sample(frac=1, random_state=random_seed)
            .reset_index(drop=True)
        )

        val_set = (
            df_val.drop(columns=["T_bins", "kL_bins", "Strata"])
            .sample(frac=1, random_state=random_seed)
            .reset_index(drop=True)
        )

        test_set = (
            df_test.drop(columns=["T_bins", "kL_bins", "Strata"])
            .sample(frac=1, random_state=random_seed)
            .reset_index(drop=True)
        )

        print("Train shape:", train_set.shape)

        X_val = val_set.iloc[:, 1:-2].to_numpy()
        X_train = train_set.iloc[:, 1:-2].to_numpy()
        X_test = test_set.iloc[:, 1:-2].to_numpy()
        X_ext = df_ext.iloc[:, 1:-2].to_numpy()
        y_val = val_set.iloc[:, -2].to_numpy()
        y_train = train_set.iloc[:, -2].to_numpy()
        y_test = test_set.iloc[:, -2].to_numpy()
        y_ext = df_ext.iloc[:, -2].to_numpy()

        split_dict = {
            "X_train": X_train,
            "X_val": X_val,
            "X_test": X_test,
            "X_ext": X_ext,
            "y_train": y_train,
            "y_val": y_val,
            "y_test": y_test,
            "y_ext": X_ext,
        }

        with open(run_dir / "dataset_split.pkl", "wb") as f:
            pickle.dump(split_dict, f, protocol=pickle.HIGHEST_PROTOCOL)

        in_features = X_train.shape[1]
        print(f"Input features for model: {in_features}")

        train_loader, val_loader, test_loader = create_dataloaders(
            X_train,
            X_val,
            X_test,
            y_train,
            y_val,
            y_test,
            batch_size=CONFIG["batch_size"],
            shuffle_train=CONFIG["shuffle_train"],
        )

        ext_dataset = data_utils.RegressionDataset(X_ext, y_ext)
        ext_loader = data_utils.DataLoader(
            ext_dataset, batch_size=CONFIG["batch_size"], shuffle=False
        )
        dataloaders = {
            "train": train_loader,
            "val": val_loader,
            "test": test_loader,
            "ext": ext_loader,
        }

        for model_idx in range(1, 6):
            print(f"\n--- Training model {model_idx} on Run {run_idx} ---")
            set_seed(42 * (run_idx * 100) + model_idx)

            model_dir = run_dir / f"model_{model_idx}"
            model_dir.mkdir(parents=True, exist_ok=True)
            best_model_path = model_dir / f"models/ffnn_model_best.pth"

            if best_model_path.exists():
                print("Best model found — skipping training.")

            else:

                model = FFNN(
                    in_features,
                    CONFIG["model_params"]["n_layers"],
                    CONFIG["model_params"]["act_fn"],
                    CONFIG["model_params"]["num_neu_list"],
                    CONFIG["model_params"]["p"],
                )
                model.apply(lambda m: init_weights(m, nonlinearity=model.act_fn_name))
                model = model.to(device)

                optimizer = getattr(
                    torch.optim, CONFIG["training_params"]["optimizer_name"]
                )(
                    model.parameters(),
                    lr=CONFIG["training_params"]["learning_rate"],
                    weight_decay=CONFIG["training_params"]["weight_decay"],
                )
                scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
                    optimizer,
                    mode="min",
                    factor=0.5,
                    patience=CONFIG["training_params"]["scheduler_patience"],
                )

                early_stopper = EarlyStopper(
                    patience=CONFIG["training_params"]["early_stopping_patience"],
                    delta=CONFIG["training_params"]["early_stopping_delta"],
                    minimize=True,
                )

                model_dir = run_dir / f"model_{model_idx}"
                model_dir.mkdir(parents=True, exist_ok=True)

                best_val_rmse, best_model_path = train_and_val_proc(
                    model,
                    optimizer,
                    evidential_regression,
                    CONFIG["training_params"]["epochs"],
                    scheduler,
                    dataloaders,
                    early_stopper,
                    device,
                    model_dir,
                    lamb_loss=CONFIG["training_params"]["loss_lamb"],
                    target_inverse_transform=(
                        CONFIG["target_transform"]["inverse_function_for_metrics"]
                        if CONFIG["target_transform"]["enabled"]
                        else None
                    ),
                )

                print(f"Model {model_idx} best validation RMSE: {best_val_rmse:.3f}")

            final_model = FFNN(
                in_features,
                CONFIG["model_params"]["n_layers"],
                CONFIG["model_params"]["act_fn"],
                CONFIG["model_params"]["num_neu_list"],
                CONFIG["model_params"]["p"],
            )
            final_model.load_state_dict(
                torch.load(best_model_path, map_location=device)
            )
            final_model.to(device)

            test_proc(
                final_model,
                nig_nll,
                dataloaders,
                device,
                model_dir,
                target_inverse_transform=(
                    CONFIG["target_transform"]["inverse_function_for_metrics"]
                    if CONFIG["target_transform"]["enabled"]
                    else None
                ),
            )
            test_proc(
                final_model,
                nig_nll,
                dataloaders,
                device,
                model_dir,
                target_inverse_transform=(
                    CONFIG["target_transform"]["inverse_function_for_metrics"]
                    if CONFIG["target_transform"]["enabled"]
                    else None
                ),
                key="ext",
            )
            val_proc(
                final_model,
                nig_nll,
                dataloaders,
                device,
                model_dir,
                target_inverse_transform=(
                    CONFIG["target_transform"]["inverse_function_for_metrics"]
                    if CONFIG["target_transform"]["enabled"]
                    else None
                ),
            )

In [ ]:
run_experiment(
    df_all,
    df_telab,
    10,
    f"./new_experiment_results",
)

## 4-2- General Assessment

In [ ]:
def evaluate_uncertainty(
    y_true: np.ndarray, y_pred: np.ndarray, total_unc: np.ndarray, num_bins: int = 50
) -> dict:
    """
    Evaluate uncertainty calibration and quality metrics.

    Parameters
    ----------
    y_true : np.ndarray
        Ground truth target values. Shape (N,).
    y_pred : np.ndarray
        Predicted mean values. Shape (N,).
    total_unc : np.ndarray
        Predicted total uncertainties (variances). Shape (N,).
    num_bins : int, optional
        Number of confidence levels to evaluate miscalibration area, by default 50.

    Returns
    -------
    dict
        Dictionary with:
        - 'NLL': Negative log-likelihood
        - 'MiscalibrationArea': Area between ideal and empirical coverage
        - 'SpearmanR': Spearman rank correlation between abs error and uncertainty
    """
    y_true, y_pred, total_unc = (
        np.asarray(y_true),
        np.asarray(y_pred),
        np.asarray(total_unc),
    )

    nll = 0.5 * np.mean(
        np.log(2 * np.pi * total_unc) + ((y_true - y_pred) ** 2) / total_unc
    )

    sigmas = np.sqrt(total_unc)
    abs_error = np.abs(y_true - y_pred)
    alpha_levels = np.linspace(0.05, 0.95, num_bins)
    z_scores = norm.ppf(0.5 + alpha_levels / 2)

    empirical_coverages = []
    for z in z_scores:
        inside = (abs_error <= z * sigmas).mean()
        empirical_coverages.append(inside)

    ideal_coverages = alpha_levels
    miscal_area = np.trapz(
        np.abs(np.array(empirical_coverages) - ideal_coverages), alpha_levels
    )

    rho, _ = spearmanr(abs_error, np.sqrt(total_unc))

    return {"NLL": nll, "MiscalibrationArea": miscal_area, "SpearmanR": rho}

In [ ]:
total_fv_df = pd.read_csv("../feature_vectors/all_data.csv")

In [ ]:
print("Dataset -> number, unique:")
for name in total_fv_df["name"].unique():
    print(f"{name}:")
    num = total_fv_df[total_fv_df["name"] == name].shape[0]
    num_uniq = total_fv_df[total_fv_df["name"] == name]["Formula"].unique().shape[0]
    min_kl = total_fv_df[total_fv_df["name"] == name]["kL"].min()
    max_kl = total_fv_df[total_fv_df["name"] == name]["kL"].max()
    min_T = total_fv_df[total_fv_df["name"] == name]["T"].min()
    max_T = total_fv_df[total_fv_df["name"] == name]["T"].max()
    print(f"# {num}({num_uniq})")
    print(f"kL: {min_kl}-{max_kl}")
    print(f"T: {min_T}-{max_T}")

In [ ]:
root_dir = "./new_experiment_results/"

all_results = []

n_split = 1

for split in sorted(os.listdir(root_dir)):

    split_path = os.path.join(root_dir, split)
    if os.path.isdir(split_path) and split.startswith("run_"):

        n_model = 1
        for model in sorted(os.listdir(split_path)):
            model_path = os.path.join(split_path, model)
            if os.path.isdir(model_path) and model.startswith("model_"):

                test_file = os.path.join(model_path, "results_ffnn_model_test_set.csv")

                if os.path.exists(test_file):
                    df = pd.read_csv(test_file)

                    df["split"] = n_split
                    df["model"] = n_model

                    all_results.append(df)
                    n_model += 1

        n_split += 1

if all_results:
    aggregated_results = pd.concat(all_results, ignore_index=True)

    aggregated_results.to_csv(
        "./new_experiment_results/aggregated_test_results.csv", index=False
    )
    print("Aggregated results saved to 'aggregated_test_results.csv'")
else:
    print("No test results found!")

In [ ]:
root_dir = "./new_experiment_results"
all_results = []

n_split = 1

for split in sorted(os.listdir(root_dir)):
    split_path = os.path.join(root_dir, split)
    if os.path.isdir(split_path) and split.startswith("run_"):

        n_model = 1
        for model in sorted(os.listdir(split_path)):
            model_path = os.path.join(split_path, model)
            if os.path.isdir(model_path) and model.startswith("model_"):

                test_file = os.path.join(model_path, "results_enn_test_set.csv")

                if os.path.exists(test_file):
                    unc_df = pd.read_csv(test_file)

                    metrics = evaluate_uncertainty(
                        y_true=unc_df["y_test"].to_numpy(),
                        y_pred=unc_df["y_pred"].to_numpy(),
                        total_unc=unc_df["total_unc"].to_numpy(),
                    )

                    all_results.append(
                        {
                            "split": n_split,
                            "model": n_model,
                            "NLL": metrics["NLL"],
                            "MCA": metrics["MiscalibrationArea"],
                            "SpearmanR": metrics["SpearmanR"],
                        }
                    )

                    n_model += 1

        n_split += 1

if all_results:
    aggregated_results = pd.DataFrame(all_results)
    output_path = "./new_experiment_results/unc_aggregated_test_metrics.csv"
    aggregated_results.to_csv(output_path, index=False)
    print(f"Aggregated results saved to '{output_path}'")
else:
    print("No test results found!")

In [ ]:
per_metrics_df = pd.read_csv("./new_experiment_results/aggregated_test_results.csv")
un_metrics_df = pd.read_csv("./new_experiment_results/unc_aggregated_test_metrics.csv")

In [ ]:
per_metrics_df.iloc[:, 1:].describe().iloc[1:3, :-2].round(3)

In [ ]:
un_metrics_df.iloc[:, 2:].describe().iloc[1:3, :].round(3)

## 4-3- Plots

### 4-3-1- kL Distribution

In [ ]:
all_data_kl = pd.read_csv("../feature_vectors/all_data.csv")["kL"]
starrydataset_kl = pd.read_csv("../dataset/clean_starrydataset.csv")["k"]

In [ ]:
font_manager.findfont("Helvetica Light")
plt.rc("font", family="Helvetica Light")
plt.rc("font", serif="Helvetica Light", size=28)
plt.rcParams["axes.linewidth"] = 1.5
plt.rcParams["xtick.major.size"] = 8
plt.rcParams["xtick.major.width"] = 1.5
plt.rcParams["ytick.major.size"] = 8
plt.rcParams["ytick.major.width"] = 1.5
plt.rcParams["ytick.direction"] = "in"
plt.rcParams["xtick.direction"] = "in"
plt.rcParams["legend.markerscale"] = 2

plt.rcParams["mathtext.it"] = "Helvetica Light:italic"
plt.rcParams["mathtext.rm"] = "Helvetica Light"
plt.rcParams["mathtext.default"] = "regular"

plt.figure(figsize=(10, 8))

LW = 1
FILL = True
COLORS = ["#084c61", "#db504a"]
ALPHAS = [0.5, 0.5]

sns.kdeplot(
    all_data_kl,
    label="Aggregated Data",
    linewidth=LW,
    color=COLORS[0],
    fill=FILL,
    alpha=ALPHAS[0],
)
sns.kdeplot(
    starrydataset_kl,
    label="Starrydata",
    linewidth=LW,
    color=COLORS[1],
    fill=FILL,
    alpha=ALPHAS[1],
)


plt.xlabel("$\kappa_{L}$ Value")
plt.ylabel("Density")
plt.legend(
    fontsize=22,
    frameon=True,
    framealpha=0.5,
    labelspacing=0.3,
    handlelength=1.5,
    borderaxespad=0.4,
    handletextpad=0.4,
    borderpad=0.4,
)
plt.tight_layout()
plt.savefig("../figures/kl_distribution.png", dpi=300, bbox_inches="tight")
plt.show()

### 4-3-2- Element Distribution

In [ ]:
df1 = pd.read_csv("../feature_vectors/all_data.csv")
df1["Formula"].to_excel("../dataset/CAF_input_formula_all_data.xlsx")

generate_periodic_table_heatmap(
    input_xlsx="../dataset/CAF_input_formula_all_data.xlsx",
    cache_xlsx="../dataset/CAF_input_formula_all_data_elements_sorted_element_count.xlsx",
    output_png="../figures/CAF_input_formula_all_data_elements_sorted_ptable.png",
    heatmap_colors=["#084c6175", "#db4f4a9b"],
)

df2 = pd.read_csv("../dataset/clean_starrydataset.csv")
df2.rename({"iupac_notation": "Formula"}, axis=1)["Formula"].to_excel(
    "../dataset/CAF_input_formula_starrydata.xlsx"
)
generate_periodic_table_heatmap(
    input_xlsx="../dataset/CAF_input_formula_starrydata.xlsx",
    cache_xlsx="../dataset/CAF_input_formula_starrydata_elements_sorted_element_count.xlsx",
    output_png="../figures/CAF_input_formula_starrydata_elements_sorted_ptable.png",
    heatmap_colors=["#084c6175", "#db4f4a9b"],
)

In [ ]:
GAP = 10
FONT_PATH = "/usr/share/fonts/truetype/helvetica-255/helvetica-light-587ebe5a59211.ttf"
# FONT_PATH = "C:/Users/Milad/AppData/Local/Microsoft/Windows/Fonts/Helvetica-Light-587ebe5a59211.ttf"
BACKGROUND_COLOR = (255, 255, 255)

img_a = Image.open(
    "../figures/CAF_input_formula_all_data_elements_sorted_ptable.png"
).convert("RGB")
img_b = Image.open("../figures/kl_distribution.png").convert("RGB")

SCALE_A = 1.10
new_width_a = int(img_a.width * SCALE_A)
img_a = img_a.resize((new_width_a, img_a.height), Image.LANCZOS)

if img_a.height != img_b.height:
    new_width = int(img_b.width * img_a.height / img_b.height)
    img_b = img_b.resize((new_width, img_a.height), Image.LANCZOS)

FONT_SIZE = 150
LABEL_HEIGHT = 400
font = ImageFont.truetype(FONT_PATH, FONT_SIZE)

canvas = Image.new(
    "RGB",
    (img_a.width + GAP + img_b.width, img_a.height + LABEL_HEIGHT),
    BACKGROUND_COLOR,
)
canvas.paste(img_a, (0, 0))
canvas.paste(img_b, (img_a.width + GAP, 0))

draw = ImageDraw.Draw(canvas)
LABEL_Y = img_a.height + (LABEL_HEIGHT - FONT_SIZE) // 2
for label, x_center in [
    ("(a)", img_a.width // 2),
    ("(b)", img_a.width + GAP + img_b.width // 2),
]:
    w = draw.textbbox((0, 0), label, font=font)[2]
    draw.text((x_center - w // 2, LABEL_Y), label, fill=(0, 0, 0), font=font)

canvas.save("../figures/element_kl_distributions.png", dpi=(300, 300))
canvas.show()

### 4-3-3- Performance + Scatter Plot

In [ ]:
mean_metrics = [0.810, 0.898, 0.523, 0.594, 0.930, 0.098, 0.584]
std_metrics = [0.101, 0.056, 0.035, 0.047, 0.188, 0.018, 0.042]
label_metrics = [
    r"MSE",
    r"RMSE",
    r"MAE",
    r"$\rm R^{2}$",
    r"NLL",
    r"MCA",
    r"$\rm R_{S}$",
]

yerr_upper = np.vstack([np.zeros_like(std_metrics), std_metrics])

root_dir = "./new_experiment_results"

all_re_results = []
all_unc_results = []

for split in sorted(os.listdir(root_dir)):
    split_path = os.path.join(root_dir, split)
    if os.path.isdir(split_path) and split.startswith("run_"):

        for model in sorted(os.listdir(split_path)):
            model_path = os.path.join(split_path, model)
            if os.path.isdir(model_path) and model.startswith("model_"):

                test_file = os.path.join(model_path, "results_enn_test_set.csv")

                unc_df = pd.read_csv(test_file)
                y_true = unc_df["y_test"].to_numpy()
                y_pred = unc_df["y_pred"].to_numpy()
                residual_error = abs(y_true - y_pred)
                total_unc = unc_df["total_unc"].to_numpy()

                all_re_results.append(residual_error)
                all_unc_results.append(total_unc)

        if split == "run_1":
            break

re_result = np.concatenate(all_re_results).reshape(-1, 5).mean(axis=1)
unc_result = np.concatenate(all_unc_results).reshape(-1, 5).mean(axis=1)

with open("../figures/residual_errors.npy", "wb") as f:
    np.save(f, re_result, allow_pickle=False)

with open("../figures/uncertainty_values.npy", "wb") as f:
    np.save(f, unc_result, allow_pickle=False)

font_manager.findfont("Helvetica Light")
plt.rc("font", family="Helvetica Light")
plt.rc("font", serif="Helvetica Light", size=24)

plt.rcParams["ytick.direction"] = "in"
plt.rcParams["xtick.direction"] = "in"
plt.rcParams["text.usetex"] = False
plt.rcParams["mathtext.it"] = "Helvetica Light:italic"
plt.rcParams["mathtext.rm"] = "Helvetica Light"
plt.rcParams["mathtext.default"] = "regular"
plt.rcParams["axes.linewidth"] = 1.25
plt.rcParams["xtick.major.size"] = 3
plt.rcParams["xtick.minor.size"] = 0
plt.rcParams["xtick.major.width"] = 1.25
plt.rcParams["xtick.minor.width"] = 0
plt.rcParams["ytick.major.size"] = 3
plt.rcParams["ytick.minor.size"] = 0
plt.rcParams["ytick.major.width"] = 1.25
plt.rcParams["ytick.minor.width"] = 0

COLORS = ["#084c61", "#db4f4aca"]
LINEWIDTH = 2
BAR_WIDTH = 0.7
CAP_SIZE = 5
SCATTER_SIZE = 120
LABEL_PAD = 15

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, 8))

ax1.bar(
    x=label_metrics,
    height=mean_metrics,
    yerr=yerr_upper,
    width=BAR_WIDTH,
    color=COLORS[1],
    edgecolor=COLORS[0],
    linewidth=LINEWIDTH,
    hatch="/",
    error_kw={"capsize": CAP_SIZE, "capthick": LINEWIDTH, "elinewidth": LINEWIDTH},
)
ax1.set_xlabel("(a) Metrics", labelpad=LABEL_PAD)
ax1.set_ylabel("Value")
ax1.set_xticklabels(label_metrics)
ax1.set_ylim(0, 1.2)

ax2.scatter(
    re_result,
    unc_result,
    c=COLORS[1],
    edgecolors=COLORS[0],
    s=SCATTER_SIZE,
    linewidths=LINEWIDTH,
)
ax2.set_xlabel("(b) Residual Error", labelpad=LABEL_PAD)
ax2.set_ylabel("Total Uncertainty")
ax2.set_xlim(0.01, 2.0)
ax2.set_ylim(0, 3.0)

plt.tight_layout()
plt.savefig("../figures/combined_metrics_parity.png", dpi=300, bbox_inches="tight")
plt.show()

### 4-3-5 Error vs Confidence Plot

In [ ]:
def cutoff_errors_percentile(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    unc: np.ndarray,
) -> pd.DataFrame:
    """
    Compute MAE, MSE, RMSE, and Pearson correlation between uncertainty and error
    as a function of confidence percentile.

    Parameters
    ----------
    y_true : np.ndarray
        Ground-truth values of shape (n,).
    y_pred : np.ndarray
        Predicted values of shape (n,).
    unc : np.ndarray
        Uncertainty estimates of shape (n,).
        Lower values correspond to higher confidence.

    Returns
    -------
    pd.DataFrame
        DataFrame containing:
        - percentile
        - mae
        - mse
        - rmse
        - pearson_corr
    """

    n = len(y_true)
    percentiles = np.arange(0.10, 1.01, 0.01)

    mae = np.zeros_like(percentiles, dtype=float)
    mse = np.zeros_like(percentiles, dtype=float)
    rmse = np.zeros_like(percentiles, dtype=float)
    pearson = np.zeros_like(percentiles, dtype=float)

    idx = np.argsort(unc)

    for i, p in enumerate(percentiles):
        k = max(1, int(p * n))
        selected = idx[:k]

        residuals = y_true[selected] - y_pred[selected]
        abs_err = np.abs(residuals)

        mae[i] = np.mean(abs_err)
        mse[i] = np.mean(residuals**2)
        rmse[i] = np.sqrt(mse[i])

        if k > 1 and np.std(unc[selected]) > 0 and np.std(abs_err) > 0:
            pearson[i] = np.corrcoef(unc[selected], abs_err)[0, 1]
        else:
            pearson[i] = np.nan

    return pd.DataFrame(
        {
            "percentile": percentiles,
            "mae": mae,
            "mse": mse,
            "rmse": rmse,
            "rp": pearson,
        }
    )


def aggregate_percentile_metrics(dfs: List[pd.DataFrame]) -> pd.DataFrame:
    """
    Compute mean and std of mae, mse, rmse, and rp across a list of
    percentile-aligned DataFrames.

    Parameters
    ----------
    dfs : list of pd.DataFrame
        Each dataframe must contain columns:
        ['percentile', 'mae', 'mse', 'rmse', 'rp'],
        all with identical percentile rows.

    Returns
    -------
    pd.DataFrame
        Columns:
        percentile, mae_mean, mae_std, mse_mean, mse_std,
        rmse_mean, rmse_std, rp_mean, rp_std
    """

    full = pd.concat(dfs, axis=0, ignore_index=True)

    grouped = full.groupby("percentile")

    out = pd.DataFrame(
        {
            "percentile": grouped["percentile"].first(),
            "mae_mean": grouped["mae"].mean(),
            "mae_std": grouped["mae"].std(),
            "mse_mean": grouped["mse"].mean(),
            "mse_std": grouped["mse"].std(),
            "rmse_mean": grouped["rmse"].mean(),
            "rmse_std": grouped["rmse"].std(),
            "rp_mean": grouped["rp"].mean(),
            "rp_std": grouped["rp"].std(),
        }
    ).reset_index(drop=True)

    return out

In [ ]:
root_dir = "./new_experiment_results"

all_results = []

for split in sorted(os.listdir(root_dir)):
    split_path = os.path.join(root_dir, split)
    if os.path.isdir(split_path) and split.startswith("run_"):

        for model in sorted(os.listdir(split_path)):
            if model.startswith("model_"):
                model_path = os.path.join(split_path, model)

                cal_un_df = pd.read_csv(
                    os.path.join(model_path, "results_enn_test_set.csv"),
                )
                all_results.append(cal_un_df)

In [ ]:
total_unc_dfs = [
    cutoff_errors_percentile(
        cal_un_df["y_test"], cal_un_df["y_pred"], cal_un_df["total_unc"]
    )
    for cal_un_df in all_results
]
epistemic_unc_dfs = [
    cutoff_errors_percentile(
        cal_un_df["y_test"], cal_un_df["y_pred"], cal_un_df["epistemic_unc"]
    )
    for cal_un_df in all_results
]
aleatoric_unc_dfs = [
    cutoff_errors_percentile(
        cal_un_df["y_test"], cal_un_df["y_pred"], cal_un_df["aleatoric_unc"]
    )
    for cal_un_df in all_results
]

aggregated_total_unc_df = aggregate_percentile_metrics(total_unc_dfs)
aggregated_epistemic_unc_df = aggregate_percentile_metrics(epistemic_unc_dfs)
aggregated_aleatoric_unc_df = aggregate_percentile_metrics(aleatoric_unc_dfs)

In [ ]:
aggregated_total_unc_df.to_csv(
    "../figures/general_assessment_total_unc.csv", index=False
)
aggregated_epistemic_unc_df.to_csv(
    "../figures/general_assessment_epistemic_unc.csv", index=False
)
aggregated_aleatoric_unc_df.to_csv(
    "../figures/general_assessment_aleatoric_unc.csv", index=False
)

In [ ]:
aggregated_total_unc_df = pd.read_csv("../figures/general_assessment_total_unc.csv")
aggregated_epistemic_unc_df = pd.read_csv(
    "../figures/general_assessment_epistemic_unc.csv"
)
aggregated_aleatoric_unc_df = pd.read_csv(
    "../figures/general_assessment_aleatoric_unc.csv"
)

font_manager.findfont("Helvetica Light")
plt.rc("font", family="Helvetica Light")
plt.rc("font", serif="Helvetica Light", size=26)
plt.rcParams["axes.linewidth"] = 1.5
plt.rcParams["xtick.major.size"] = 8
plt.rcParams["xtick.major.width"] = 1.5
plt.rcParams["ytick.major.size"] = 8
plt.rcParams["ytick.major.width"] = 1.5
plt.rcParams["ytick.direction"] = "in"
plt.rcParams["xtick.direction"] = "in"
plt.rcParams["legend.markerscale"] = 2

plt.rcParams["mathtext.it"] = "Helvetica Light:italic"
plt.rcParams["mathtext.rm"] = "Helvetica Light"
plt.rcParams["mathtext.default"] = "regular"

color_total = "#084c61d4"
color_epistemic = "#db504ad4"
color_aleatoric = "#e3b705d4"
line_width = 4
fill_alpha = 0.1
label_pad = 15

fig, axes = plt.subplots(1, 2, figsize=(20, 8))

p = aggregated_total_unc_df["percentile"]
m = aggregated_total_unc_df["rmse_mean"]
s = aggregated_total_unc_df["rmse_std"]
axes[0].plot(p, m, label="Total", color=color_total, linewidth=line_width)
axes[0].fill_between(p, m - s, m + s, color=color_total, alpha=fill_alpha)

p = aggregated_epistemic_unc_df["percentile"]
m = aggregated_epistemic_unc_df["rmse_mean"]
s = aggregated_epistemic_unc_df["rmse_std"]
axes[0].plot(p, m, label="Epistemic", color=color_epistemic, linewidth=line_width)
axes[0].fill_between(p, m - s, m + s, color=color_epistemic, alpha=fill_alpha)

p = aggregated_aleatoric_unc_df["percentile"]
m = aggregated_aleatoric_unc_df["rmse_mean"]
s = aggregated_aleatoric_unc_df["rmse_std"]
axes[0].plot(p, m, label="Aleatoric", color=color_aleatoric, linewidth=line_width)
axes[0].fill_between(p, m - s, m + s, color=color_aleatoric, alpha=fill_alpha)

axes[0].set_xlabel("1-Confidence Percentile", labelpad=label_pad)
axes[0].set_ylabel("RMSE")
axes[0].set_yticks(np.arange(0.1, 1.0, 0.2))
axes[0].set_xticks(np.arange(0.0, 1.2, 0.2))

p = aggregated_total_unc_df["percentile"]
m = aggregated_total_unc_df["mae_mean"]
s = aggregated_total_unc_df["mae_std"]
axes[1].plot(p, m, label="Total", color=color_total, linewidth=line_width)
axes[1].fill_between(p, m - s, m + s, color=color_total, alpha=fill_alpha)

p = aggregated_epistemic_unc_df["percentile"]
m = aggregated_epistemic_unc_df["mae_mean"]
s = aggregated_epistemic_unc_df["mae_std"]
axes[1].plot(p, m, label="Epistemic", color=color_epistemic, linewidth=line_width)
axes[1].fill_between(p, m - s, m + s, color=color_epistemic, alpha=fill_alpha)

p = aggregated_aleatoric_unc_df["percentile"]
m = aggregated_aleatoric_unc_df["mae_mean"]
s = aggregated_aleatoric_unc_df["mae_std"]
axes[1].plot(p, m, label="Aleatoric", color=color_aleatoric, linewidth=line_width)
axes[1].fill_between(p, m - s, m + s, color=color_aleatoric, alpha=fill_alpha)

axes[1].set_xlabel("1-Confidence Percentile", labelpad=label_pad)
axes[1].set_ylabel("MAE")
axes[1].set_yticks(np.arange(0.1, 0.6, 0.1))
axes[1].set_xticks(np.arange(0.0, 1.2, 0.2))

axes[1].legend(loc="upper left", frameon=True)

plt.tight_layout()
plt.subplots_adjust(wspace=0.13)
plt.savefig("../figures/error_confidence_percentile.png", dpi=300, bbox_inches="tight")
plt.show()

### 4-3-6- Calibration Curve

In [ ]:
root_dir = "./new_experiment_results"

all_results = []

for split in sorted(os.listdir(root_dir)):
    split_path = os.path.join(root_dir, split)
    if os.path.isdir(split_path) and split.startswith("run_"):

        for model in sorted(os.listdir(split_path)):
            if model.startswith("model_"):
                model_path = os.path.join(split_path, model)

                cal_un_df = pd.read_csv(
                    os.path.join(model_path, "results_enn_test_set.csv"),
                )
                all_results.append(cal_un_df)

pd.to_pickle(all_results, "../figures/miscalibration_fig.pkl")
all_results = pd.read_pickle("../figures/miscalibration_fig.pkl")

font_manager.findfont("Helvetica Light")
plt.rc("font", family="Helvetica Light", size=24)
plt.rcParams["axes.linewidth"] = 1.5
plt.rcParams["xtick.major.size"] = 8
plt.rcParams["xtick.major.width"] = 1.5
plt.rcParams["ytick.major.size"] = 8
plt.rcParams["ytick.major.width"] = 1.5
plt.rcParams["ytick.direction"] = "in"
plt.rcParams["xtick.direction"] = "in"
plt.rcParams["legend.markerscale"] = 2
plt.rcParams["mathtext.it"] = "Helvetica Light:italic"
plt.rcParams["mathtext.rm"] = "Helvetica Light"
plt.rcParams["mathtext.default"] = "regular"

color_total = "#084c61d4"
color_epistemic = "#db504ad4"
color_aleatoric = "#e3b705d4"
LW = 4

total_obs_list = []
epistemic_obs_list = []
aleatoric_obs_list = []

for df in all_results:
    y_pred = df["y_pred"].to_numpy()
    y_test = df["y_test"].to_numpy()

    total_unc = np.sqrt(df["total_unc"].to_numpy())
    epistemic_unc = np.sqrt(df["epistemic_unc"].to_numpy())
    aleatoric_unc = np.sqrt(df["aleatoric_unc"].to_numpy())

    total_exp, total_obs = get_proportion_lists(y_pred, total_unc, y_test)
    total_obs_list.append(total_obs)

    epistemic_exp, epistemic_obs = get_proportion_lists(y_pred, epistemic_unc, y_test)
    epistemic_obs_list.append(epistemic_obs)

    aleatoric_exp, aleatoric_obs = get_proportion_lists(y_pred, aleatoric_unc, y_test)
    aleatoric_obs_list.append(aleatoric_obs)

exp_proportions = total_exp

total_obs_arr = np.array(total_obs_list)
epistemic_obs_arr = np.array(epistemic_obs_list)
aleatoric_obs_arr = np.array(aleatoric_obs_list)

total_mean = np.mean(total_obs_arr, axis=0)
total_std = np.std(total_obs_arr, axis=0)
epistemic_mean = np.mean(epistemic_obs_arr, axis=0)
epistemic_std = np.std(epistemic_obs_arr, axis=0)
aleatoric_mean = np.mean(aleatoric_obs_arr, axis=0)
aleatoric_std = np.std(aleatoric_obs_arr, axis=0)

plt.figure(figsize=(10, 10))
plt.plot([0, 1], [0, 1], "k--", label="Ideal")

plt.plot(exp_proportions, total_mean, color=color_total, label="Total", lw=LW)
plt.plot(
    exp_proportions, epistemic_mean, color=color_epistemic, label="Epistemic", lw=LW
)
plt.plot(
    exp_proportions, aleatoric_mean, color=color_aleatoric, label="Aleatoric", lw=LW
)

plt.fill_between(
    exp_proportions,
    total_mean - total_std,
    total_mean + total_std,
    alpha=0.12,
    color=color_total,
)
plt.fill_between(
    exp_proportions,
    epistemic_mean - epistemic_std,
    epistemic_mean + epistemic_std,
    alpha=0.12,
    color=color_epistemic,
)
plt.fill_between(
    exp_proportions,
    aleatoric_mean - aleatoric_std,
    aleatoric_mean + aleatoric_std,
    alpha=0.12,
    color=color_aleatoric,
)

plt.xlabel("Predicted Proportion in Interval")
plt.ylabel("Observed Proportion in Interval")
plt.xlim(-0.02, 1.02)
plt.ylim(-0.02, 1.02)
plt.legend(loc="upper left")
plt.savefig("../figures/miscalibration_plot.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
pd.to_pickle(all_results, "../figures/miscalibration_fig.pkl")
loaded_list = pd.read_pickle("../figures/miscalibration_fig.pkl")

### 4-3-7- Epistemic and Aleatoric Uncertainties for High and Low kL

In [ ]:
high_kl_epist = []
high_kl_aleat = []

low_kl_epist = []
low_kl_aleat = []

root_dir = "./new_experiment_results"
run = 0
for split in sorted(os.listdir(root_dir)):
    split_path = os.path.join(root_dir, split)

    all_results = []
    if os.path.isdir(split_path) and split.startswith("run_"):
        run += 1

        for model in sorted(os.listdir(split_path)):
            model_path = os.path.join(split_path, model)

            if os.path.isdir(model_path) and model.startswith("model_"):

                test_file = os.path.join(model_path, "results_enn_test_set.csv")

                if os.path.exists(test_file):
                    df = pd.read_csv(test_file)
                    all_results.append(df)

        results = pd.DataFrame(
            np.mean(all_results, axis=0),
            columns=["y_test", "y_pred", "aleatoric", "epistemic", "total"],
        )
        results["residual"] = abs(results["y_test"] - results["y_pred"])
        results["nor_epist"] = results["epistemic"] / results["total"]
        results["nor_aleat"] = results["aleatoric"] / results["total"]

        results.sort_values("y_test", inplace=True)

        q1, q3 = results.describe().iloc[[4, 6], 0]

        print(f"Run {run}: Q1 (kL low): {q1:.3f} / Q3 (kL high): {q3:.3f}")

        low_kl = results.iloc[:100, :]
        high_kl = results.iloc[300:400, :]

        nor_epist, nor_aleat = low_kl.describe().iloc[1, -2:]
        low_kl_epist.append(nor_epist)
        low_kl_aleat.append(nor_aleat)

        nor_epist, nor_aleat = high_kl.describe().iloc[1, -2:]
        high_kl_epist.append(nor_epist)
        high_kl_aleat.append(nor_aleat)

In [ ]:
df_combined = pd.DataFrame(
    {
        "low_epistemic": low_kl_epist,
        "low_aleatoric": low_kl_aleat,
        "high_epistemic": high_kl_epist,
        "high_aleatoric": high_kl_aleat,
    }
)

COLORS = ["#084c61", "#db504a"]

font_manager.findfont("Helvetica Light")
plt.rc("font", family="Helvetica Light")
plt.rc("font", serif="Helvetica Light", size=18)
plt.rcParams["axes.linewidth"] = 1.25
plt.rcParams["xtick.major.size"] = 3
plt.rcParams["xtick.minor.size"] = 0
plt.rcParams["xtick.major.width"] = 1.25
plt.rcParams["xtick.minor.width"] = 0
plt.rcParams["ytick.major.size"] = 3
plt.rcParams["ytick.minor.size"] = 0
plt.rcParams["ytick.major.width"] = 1.25
plt.rcParams["ytick.minor.width"] = 0
plt.rcParams["ytick.direction"] = "in"
plt.rcParams["xtick.direction"] = "in"
plt.rcParams["text.usetex"] = False

plt.rcParams["mathtext.it"] = "Helvetica Light:italic"
plt.rcParams["mathtext.rm"] = "Helvetica Light"
plt.rcParams["mathtext.default"] = "regular"

fig, axes = plt.subplots(1, 2, figsize=(12, 5), sharey=True)

axes[0].hist(
    [low_kl["nor_epist"], low_kl["nor_aleat"]],
    bins=10,
    label=["Epistemic", "Aleatoric"],
    alpha=0.73,
    color=[COLORS[0], COLORS[1]],
    edgecolor="black",
    linewidth=1.2,
)
axes[0].set_title("Low $\kappa_{L}$")
axes[0].set_ylabel("Frequency")
axes[0].set_xlabel("Normalized Uncertainty")

axes[1].hist(
    [high_kl["nor_epist"], high_kl["nor_aleat"]],
    bins=10,
    label=["Epistemic", "Aleatoric"],
    alpha=0.73,
    color=[COLORS[0], COLORS[1]],
    edgecolor="black",
    linewidth=1.2,
)
axes[1].set_title("High $\kappa_{L}$")
axes[1].set_xlabel("Normalized Uncertainty")
axes[1].legend()

plt.tight_layout()
plt.savefig("../figures/uncertainty_regimes.png", dpi=300, bbox_inches="tight")
plt.show()

### 4-3-8- Scatter Plot (Predicted vs True)

In [ ]:
root_dir = "./new_experiment_results"
all_results = []

n_split = 1

for split in sorted(os.listdir(root_dir)):
    split_path = os.path.join(root_dir, split)
    if os.path.isdir(split_path) and split.startswith("run_"):

        n_model = 1
        for model in sorted(os.listdir(split_path)):
            model_path = os.path.join(split_path, model)
            if os.path.isdir(model_path) and model.startswith("model_"):

                test_file = os.path.join(model_path, "results_enn_test_set.csv")

                if os.path.exists(test_file):
                    result_df = pd.read_csv(test_file)
                    all_results.append(result_df)
        break

In [ ]:
combined_result_df = pd.concat(all_results)
mean_result_df = combined_result_df.groupby(level=0).mean()
std_result_df = combined_result_df.groupby(level=0).std()

In [ ]:
mean_result_df["nor_epist"] = (
    mean_result_df["epistemic_unc"] / mean_result_df["total_unc"]
)
mean_result_df["nor_aleat"] = (
    mean_result_df["aleatoric_unc"] / mean_result_df["total_unc"]
)

plt.rc("font", family="Helvetica Light", serif="Helvetica Light", size=20)
plt.rcParams["axes.linewidth"] = 1.5
plt.rcParams["xtick.major.size"] = 8
plt.rcParams["xtick.major.width"] = 1.5
plt.rcParams["ytick.major.size"] = 8
plt.rcParams["ytick.major.width"] = 1.5
plt.rcParams["ytick.direction"] = "in"
plt.rcParams["xtick.direction"] = "in"
plt.rcParams["legend.markerscale"] = 2
plt.rcParams["mathtext.it"] = "Helvetica Light:italic"
plt.rcParams["mathtext.rm"] = "Helvetica Light"
plt.rcParams["mathtext.default"] = "regular"

cmap_custom = LinearSegmentedColormap.from_list("custom", ["#db504a80", "#084c6180"])


plt.figure(figsize=(10, 8))
scatter = plt.scatter(
    mean_result_df["y_test"],
    mean_result_df["y_pred"],
    c=mean_result_df["nor_epist"],
    cmap=cmap_custom,
    s=60,
    alpha=0.7,
    edgecolors="black",
    linewidths=1.4,
    vmin=0.0,
    vmax=1.0,
)

cbar = plt.colorbar(scatter)
cbar.set_label("Uncertainty", labelpad=-80)

cbar.set_ticks([0.0, 1.0])

cbar.set_ticklabels(["Aleatoric", "Epistemic"])

plt.ylabel("Predicted $\kappa_{L}$")
plt.xlabel("True $\kappa_{L}$")
plt.tight_layout()
plt.savefig("parity_plot_with_unc.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
mean_result_df["total_unc"] = (
    mean_result_df["total_unc"] - mean_result_df["total_unc"].min()
) / (mean_result_df["total_unc"].max() - mean_result_df["total_unc"].min())

plt.rc("font", family="Helvetica Light", serif="Helvetica Light", size=20)
plt.rcParams["axes.linewidth"] = 1.5
plt.rcParams["xtick.major.size"] = 8
plt.rcParams["xtick.major.width"] = 1.5
plt.rcParams["ytick.major.size"] = 8
plt.rcParams["ytick.major.width"] = 1.5
plt.rcParams["ytick.direction"] = "in"
plt.rcParams["xtick.direction"] = "in"
plt.rcParams["legend.markerscale"] = 2
plt.rcParams["mathtext.it"] = "Helvetica Light:italic"
plt.rcParams["mathtext.rm"] = "Helvetica Light"
plt.rcParams["mathtext.default"] = "regular"

cmap_custom = LinearSegmentedColormap.from_list("custom", ["#db504a80", "#084c6180"])


plt.figure(figsize=(12, 8))
scatter = plt.scatter(
    mean_result_df["y_test"],
    mean_result_df["y_pred"],
    c=mean_result_df["total_unc"],
    cmap=cmap_custom,
    s=60,
    alpha=0.7,
    edgecolors="black",
    linewidths=1.4,
    vmin=0.0,
    vmax=1.0,
)

cbar = plt.colorbar(scatter, pad=0.02)
cbar.set_label("Total Uncertainty", labelpad=0)

cbar.set_ticks([0.0, 1.0])

cbar.set_ticklabels(["0", "1"])

plt.ylabel("Predicted $\kappa_{L}$")
plt.xlabel("True $\kappa_{L}$")
plt.tight_layout()
plt.savefig("parity_plot_with_total_unc.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
mean_result_df["nor_epist"] = (
    mean_result_df["epistemic_unc"] / mean_result_df["total_unc"]
)
mean_result_df["nor_aleat"] = (
    mean_result_df["aleatoric_unc"] / mean_result_df["total_unc"]
)

mean_result_df["total_unc"] = (
    mean_result_df["total_unc"] - mean_result_df["total_unc"].min()
) / (mean_result_df["total_unc"].max() - mean_result_df["total_unc"].min())

plt.rc("font", family="Helvetica Light", serif="Helvetica Light", size=22)
plt.rcParams["axes.linewidth"] = 1.5
plt.rcParams["xtick.major.size"] = 8
plt.rcParams["xtick.major.width"] = 1.5
plt.rcParams["ytick.major.size"] = 8
plt.rcParams["ytick.major.width"] = 1.5
plt.rcParams["ytick.direction"] = "in"
plt.rcParams["xtick.direction"] = "in"
plt.rcParams["legend.markerscale"] = 2
plt.rcParams["mathtext.it"] = "Helvetica Light:italic"
plt.rcParams["mathtext.rm"] = "Helvetica Light"
plt.rcParams["mathtext.default"] = "regular"

cmap_custom = LinearSegmentedColormap.from_list("custom", ["#db504a80", "#084c6180"])

fig, axes = plt.subplots(1, 2, figsize=(18, 8), sharey=True)

ax1 = axes[0]
scatter1 = ax1.scatter(
    mean_result_df["y_test"],
    mean_result_df["y_pred"],
    c=mean_result_df["nor_epist"],
    cmap=cmap_custom,
    s=60,
    alpha=0.7,
    edgecolors="black",
    linewidths=1.4,
    vmin=0.0,
    vmax=1.0,
)

cbar1 = plt.colorbar(scatter1, ax=ax1, location="top", pad=0.02)
cbar1.set_label("Uncertainty", labelpad=0)
cbar1.set_ticks([0.0, 1.0])
cbar1.set_ticklabels(["Aleatoric", "Epistemic"])

labels1 = cbar1.ax.get_xticklabels()
labels1[0].set_ha("left")
labels1[1].set_ha("right")

ax1.set_ylabel("Predicted $\kappa_{L}$")
ax1.set_xlabel("True $\kappa_{L}$")

ax2 = axes[1]
scatter2 = ax2.scatter(
    mean_result_df["y_test"],
    mean_result_df["y_pred"],
    c=mean_result_df["total_unc"],
    cmap=cmap_custom,
    s=60,
    alpha=0.7,
    edgecolors="black",
    linewidths=1.4,
    vmin=0.0,
    vmax=1.0,
)

cbar2 = plt.colorbar(scatter2, ax=ax2, location="top", pad=0.02)
cbar2.set_label("Total Uncertainty", labelpad=0)
cbar2.set_ticks([0.0, 1.0])
cbar2.set_ticklabels(["0", "1"])

labels2 = cbar2.ax.get_xticklabels()
labels2[0].set_ha("left")
labels2[1].set_ha("right")

ax2.set_xlabel("True $\kappa_{L}$")

plt.tight_layout()
fig.subplots_adjust(wspace=0.04)
plt.savefig("combined_parity_plot_with_unc.png", dpi=300, bbox_inches="tight")
plt.show()

# 5- Epistemic and Aleatoric Uncertanties

## 5-1 -Codes

In [ ]:
def create_strata(df, n_bins=5):
    """Creates a combined stratification column based on kL and T quantiles."""

    df[f"kL_bins"] = pd.qcut(df["kL"], q=n_bins, labels=False, duplicates="drop")
    df[f"T_bins"] = pd.qcut(df["T"], q=n_bins, labels=False, duplicates="drop")

    df["Strata"] = df["T_bins"].astype(str) + "_" + df["kL_bins"].astype(str)
    return df

In [ ]:
def run_uncertainty_experiment(
    df_all: pd.DataFrame,
    col_name: str,
    runs: int,
    output_dir: Union[str, Path],
    not_target_frac: float = 0,
) -> None:
    """
    Run uncertainty experiments across multiple random seeds and model instances.

    Parameters
    ----------
    df_all : pandas.DataFrame
        Input dataset containing the target column `col_name`.
    col_name : str
        Boolean column name indicating the target class (e.g., 'Is_Chalcogenide').
    runs : int
        Number of independent experiment runs.
    output_dir : str or pathlib.Path
        Directory path for saving outputs, model checkpoints, and split files.
    not_target_frac : float, optional
        Fraction of non-target samples to include in the training set (default=0).
    """
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    output_dir = Path(output_dir)

    df_chalcogenide = df_all[df_all["Is_Chalcogenide"] == True].copy()
    df_non_chalcogenide = df_all[df_all["Is_Chalcogenide"] == False].copy()

    df_chalcogenide = create_strata(df_chalcogenide)
    df_non_chalcogenide = create_strata(df_non_chalcogenide)

    df_non_chal_rem, val_non_chal_component = train_test_split(
        df_non_chalcogenide,
        test_size=200,
        stratify=df_non_chalcogenide["Strata"],
        random_state=42,
    )

    base_train_set = df_non_chal_rem

    chal_ood_pool, df_chal_tv = train_test_split(
        df_chalcogenide,
        test_size=600,
        stratify=df_chalcogenide["Strata"],
        random_state=42,
    )

    test_set, val_chal_component = train_test_split(
        df_chal_tv, test_size=200, stratify=df_chal_tv["Strata"], random_state=42
    )

    non_target_pool = chal_ood_pool

    final_validation_set = pd.concat([val_non_chal_component, val_chal_component])
    val_set = (
        final_validation_set.drop(columns=["T_bins", "kL_bins", "Strata"])
        .sample(frac=1, random_state=42)
        .reset_index(drop=True)
    )
    test_set = (
        test_set.drop(columns=["T_bins", "kL_bins", "Strata"])
        .sample(frac=1, random_state=42)
        .reset_index(drop=True)
    )

    base_train_set = (
        base_train_set.drop(columns=["T_bins", "kL_bins", "Strata"])
        .sample(frac=1, random_state=42)
        .reset_index(drop=True)
    )

    non_target_pool = (
        non_target_pool.drop(columns=["T_bins", "kL_bins", "Strata"])
        .sample(frac=1, random_state=42)
        .reset_index(drop=True)
    )

    for run_idx in range(1, runs + 1):
        print(f"\nRun: {run_idx}")
        random_seed = 42 + run_idx

        run_dir = output_dir / f"run_{run_idx}"
        run_dir.mkdir(parents=True, exist_ok=True)

        if not_target_frac:
            extra_nontarget = non_target_pool.sample(
                frac=not_target_frac, random_state=random_seed
            )
            train_set = pd.concat([base_train_set, extra_nontarget], axis=0)

        else:
            train_set = base_train_set

        print("Train shape:", train_set.shape)

        X_val = val_set.iloc[:, 1:-3].to_numpy()
        X_train = train_set.iloc[:, 1:-3].to_numpy()
        X_test = test_set.iloc[:, 1:-3].to_numpy()
        y_val = val_set.iloc[:, -3].to_numpy()
        y_train = train_set.iloc[:, -3].to_numpy()
        y_test = test_set.iloc[:, -3].to_numpy()

        split_dict = {
            "X_train": X_train,
            "X_val": X_val,
            "X_test": X_test,
            "y_train": y_train,
            "y_val": y_val,
            "y_test": y_test,
        }

        with open(run_dir / "dataset_split.pkl", "wb") as f:
            pickle.dump(split_dict, f, protocol=pickle.HIGHEST_PROTOCOL)

        in_features = X_train.shape[1]
        print(f"Input features for model: {in_features}")

        train_loader, val_loader, test_loader = create_dataloaders(
            X_train,
            X_val,
            X_test,
            y_train,
            y_val,
            y_test,
            batch_size=CONFIG["batch_size"],
            shuffle_train=CONFIG["shuffle_train"],
        )

        dataloaders = {"train": train_loader, "val": val_loader, "test": test_loader}

        for model_idx in range(1, 6):
            print(f"\n--- Training model {model_idx} on Run {run_idx} ---")
            set_seed(42 * (run_idx * 100) + model_idx)

            model = FFNN(
                in_features,
                CONFIG["model_params"]["n_layers"],
                CONFIG["model_params"]["act_fn"],
                CONFIG["model_params"]["num_neu_list"],
                CONFIG["model_params"]["p"],
            )
            model.apply(lambda m: init_weights(m, nonlinearity=model.act_fn_name))
            model = model.to(device)

            optimizer = getattr(
                torch.optim, CONFIG["training_params"]["optimizer_name"]
            )(
                model.parameters(),
                lr=CONFIG["training_params"]["learning_rate"],
                weight_decay=CONFIG["training_params"]["weight_decay"],
            )
            scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
                optimizer,
                mode="min",
                factor=0.5,
                patience=CONFIG["training_params"]["scheduler_patience"],
            )

            early_stopper = EarlyStopper(
                patience=CONFIG["training_params"]["early_stopping_patience"],
                delta=CONFIG["training_params"]["early_stopping_delta"],
                minimize=True,
            )

            model_dir = run_dir / f"model_{model_idx}"
            model_dir.mkdir(parents=True, exist_ok=True)

            best_val_rmse, best_model_path = train_and_val_proc(
                model,
                optimizer,
                evidential_regression,
                CONFIG["training_params"]["epochs"],
                scheduler,
                dataloaders,
                early_stopper,
                device,
                model_dir,
                lamb_loss=CONFIG["training_params"]["loss_lamb"],
                target_inverse_transform=(
                    CONFIG["target_transform"]["inverse_function_for_metrics"]
                    if CONFIG["target_transform"]["enabled"]
                    else None
                ),
            )

            print(f"Model {model_idx} best validation RMSE: {best_val_rmse:.3f}")

            final_model = FFNN(
                in_features,
                CONFIG["model_params"]["n_layers"],
                CONFIG["model_params"]["act_fn"],
                CONFIG["model_params"]["num_neu_list"],
                CONFIG["model_params"]["p"],
            )
            final_model.load_state_dict(
                torch.load(best_model_path, map_location=device)
            )
            final_model.to(device)

            test_proc(
                final_model,
                nig_nll,
                dataloaders,
                device,
                model_dir,
                target_inverse_transform=(
                    CONFIG["target_transform"]["inverse_function_for_metrics"]
                    if CONFIG["target_transform"]["enabled"]
                    else None
                ),
            )

            val_proc(
                final_model,
                nig_nll,
                dataloaders,
                device,
                model_dir,
                target_inverse_transform=(
                    CONFIG["target_transform"]["inverse_function_for_metrics"]
                    if CONFIG["target_transform"]["enabled"]
                    else None
                ),
            )

In [ ]:
def process_uncertainty_results(root_dir: Union[str, Path]) -> None:
    """
    Process and aggregate uncertainty experiment results stored under a root directory.

    Parameters
    ----------
    root_dir : str or pathlib.Path
        Path to the root directory containing 'run_*' subdirectories.

    Notes
    -----
    This function performs three sequential tasks:
        1. Aggregates all 'results_ffnn_model_test_set.csv' files across runs.
        2. Computes calibrated uncertainties using 'results_enn_test_set.csv' and 'results_enn_val_set.csv'.
        3. Evaluates and aggregates uncertainty metrics from calibrated results.
    """

    root_dir = Path(root_dir)

    # Step 1: Aggregate raw FFNN test results
    all_results = []
    for split in sorted(os.listdir(root_dir)):
        split_path = root_dir / split

        if split_path.is_dir() and split.startswith("run_"):

            for model in sorted(os.listdir(split_path)):
                model_path = split_path / model
                if model_path.is_dir() and model.startswith("model_"):

                    test_file = model_path / "results_ffnn_model_test_set.csv"
                    if test_file.exists():
                        df = pd.read_csv(test_file)
                        df["split"] = split
                        df["model"] = model
                        all_results.append(df)

    if all_results:
        aggregated_results = pd.concat(all_results, ignore_index=True)
        aggregated_results.to_csv(root_dir / "aggregated_test_results.csv", index=False)
        print("Aggregated results saved to 'aggregated_test_results.csv'")
    else:
        print("No FFNN test results found!")

    # Step 2: Evaluate and aggregate calibrated uncertainty metrics
    all_metrics = []
    for split in sorted(os.listdir(root_dir)):

        split_path = root_dir / split

        if split_path.is_dir() and split.startswith("run_"):

            for model in sorted(os.listdir(split_path)):
                model_path = split_path / model

                if model_path.is_dir() and model.startswith("model_"):

                    test_file = model_path / "results_enn_test_set.csv"

                    if test_file.exists():

                        unc_df = pd.read_csv(test_file)
                        total_metrics = evaluate_uncertainty(
                            y_true=unc_df["y_test"].to_numpy(),
                            y_pred=unc_df["y_pred"].to_numpy(),
                            total_unc=unc_df["total_unc"].to_numpy(),
                        )

                        epistemic_metrics = evaluate_uncertainty(
                            y_true=unc_df["y_test"].to_numpy(),
                            y_pred=unc_df["y_pred"].to_numpy(),
                            total_unc=unc_df["epistemic_unc"].to_numpy(),
                        )

                        aleatoric_metrics = evaluate_uncertainty(
                            y_true=unc_df["y_test"].to_numpy(),
                            y_pred=unc_df["y_pred"].to_numpy(),
                            total_unc=unc_df["aleatoric_unc"].to_numpy(),
                        )

                        all_metrics.append(
                            {
                                "split": split,
                                "model": model,
                                "Total_NLL": total_metrics["NLL"],
                                "Total_MCA": total_metrics["MiscalibrationArea"],
                                "Total_SpearmanR": total_metrics["SpearmanR"],
                                "Epistemic_NLL": epistemic_metrics["NLL"],
                                "Epistemic_MCA": epistemic_metrics[
                                    "MiscalibrationArea"
                                ],
                                "Epistemic_SpearmanR": epistemic_metrics["SpearmanR"],
                                "Aleatoric_NLL": aleatoric_metrics["NLL"],
                                "Aleatoric_MCA": aleatoric_metrics[
                                    "MiscalibrationArea"
                                ],
                                "Aleatoric_SpearmanR": aleatoric_metrics["SpearmanR"],
                                "Mean_Total_Unc": unc_df["total_unc"].median(),
                                "Mean_Epistemic_Unc": unc_df["epistemic_unc"].median(),
                                "Mean_Aleatoric_Unc": unc_df["aleatoric_unc"].median(),
                            }
                        )

    if all_metrics:

        df_metrics = pd.DataFrame(all_metrics)
        output_path = root_dir / "unc_aggregated_test_metrics.csv"
        df_metrics.to_csv(output_path, index=False)
        print(f"Aggregated results saved to '{output_path.name}'")

    else:

        print("No calibrated uncertainty metrics found!")

## 5-2- Chalcogenide-based

### 5-2-1- Train and test sets

In [ ]:
df_all = pd.read_parquet("../feature_vectors/total_dataset.parquet")

In [ ]:
def is_chalcogenide(formula: str, tol_major: float = 0.1) -> bool:
    """
    Determine if a material formula corresponds to a chalcogenide.

    Parameters
    ----------
    formula : str
        Chemical formula of the material.
    tol_major : float, optional
        The minimum stoichiometric amount to consider an element “major” (default is 0.1).

    Returns
    -------
    bool
        True if one of the major elements in the composition is one of 'S', 'Se', or 'Te';
        False otherwise.
    """
    comp = Composition(formula)
    comp = comp.get_reduced_composition_and_factor()[0]
    frac = comp.get_el_amt_dict()
    chalcogens = {"S", "Se", "Te"}
    major = {el: amt for el, amt in frac.items() if amt >= tol_major}
    if not any(el in chalcogens for el in major.keys()):
        return False
    return True

In [ ]:
df_all["Is_Chalcogenide"] = df_all["Formula"].map(is_chalcogenide)

### 5-2-2- Traning Non-Chalcogenide (Train set) - Chalcogenide (Test set)

In [ ]:
for frac in [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]:

    run_uncertainty_experiment(
        df_all,
        "Is_Chalcogenide",
        10,
        f"./uncertainty_chalcogenide_results/not_target_frac_{frac}",
        not_target_frac=frac,
    )

In [ ]:
for frac in [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]:
    process_uncertainty_results(
        f"./uncertainty_chalcogenide_results/not_target_frac_{frac}"
    )

In [ ]:
all_per_metrics_df = []
all_un_metrics_df = []

for frac in [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]:

    per_metrics_path = f"./uncertainty_chalcogenide_results/not_target_frac_{frac}/aggregated_test_results.csv"
    un_metrics_path = f"./uncertainty_chalcogenide_results/not_target_frac_{frac}/unc_aggregated_test_metrics.csv"

    if frac == 0.2:
        per_metrics_df = (
            pd.read_csv(per_metrics_path)
            .drop([12, 14, 37, 48, 49], axis=0)
            .describe()
            .iloc[1:3, :]
        )
        un_metrics_df = (
            pd.read_csv(un_metrics_path)
            .drop([12, 14, 37, 48, 49], axis=0)
            .describe()
            .iloc[1:3, :]
        )

    else:
        per_metrics_df = pd.read_csv(per_metrics_path).describe().iloc[1:3, :]
        un_metrics_df = pd.read_csv(un_metrics_path).describe().iloc[1:3, :]

    per_metrics_df["non_target_frac"] = frac
    un_metrics_df["non_target_frac"] = frac

    all_per_metrics_df.append(per_metrics_df)
    all_un_metrics_df.append(un_metrics_df)

In [ ]:
pd.to_pickle(all_per_metrics_df, "../figures/calcogenide_ood_all_per_metrics.pkl")
pd.to_pickle(all_un_metrics_df, "../figures/calcogenide_ood_all_un_metrics.pkl")

In [ ]:
def plot_unc(
    ax,
    ax2,
    y_metric,
    yerr_metric,
    ylabel,
    show_legend=False,
    show_x_label=True,
    show_y_label=True,
):
    ax.errorbar(
        merged["non_target_frac"],
        merged["Mean_Epistemic_Unc_x"],
        yerr=merged["Mean_Epistemic_Unc_y"],
        fmt="o-",
        elinewidth=ELINEWIDTH,
        linewidth=LINEWIDTH,
        markersize=MARKERSIZE,
        markeredgewidth=MARKEREDGEWIDTH,
        capsize=CAPSIZE,
        capthick=CAPSTICK,
        color=COLORS[0],
        ecolor=ECOLORS[0],
        markerfacecolor=COLORS[0],
        markeredgecolor="white",
        label="Epistemic",
    )
    ax.errorbar(
        merged["non_target_frac"],
        merged["Mean_Aleatoric_Unc_x"],
        yerr=merged["Mean_Aleatoric_Unc_y"],
        fmt="s--",
        elinewidth=ELINEWIDTH,
        linewidth=LINEWIDTH,
        markersize=MARKERSIZE,
        markeredgewidth=MARKEREDGEWIDTH,
        capsize=CAPSIZE,
        capthick=CAPSTICK,
        color=COLORS[0],
        ecolor=ECOLORS[0],
        markerfacecolor=COLORS[0],
        markeredgecolor="white",
        label="Aleatoric",
    )
    ax.errorbar(
        merged["non_target_frac"],
        merged["Mean_Total_Unc_x"],
        yerr=merged["Mean_Total_Unc_y"],
        fmt="^-",
        elinewidth=ELINEWIDTH,
        linewidth=LINEWIDTH,
        markersize=MARKERSIZE,
        markeredgewidth=MARKEREDGEWIDTH,
        capsize=CAPSIZE,
        capthick=CAPSTICK,
        color=COLORS[0],
        ecolor=ECOLORS[0],
        markerfacecolor=COLORS[0],
        markeredgecolor="white",
        label="Total",
    )

    if show_legend:
        ax.legend(frameon=False, loc="upper right")
    if show_x_label:
        ax.set_xlabel("Chalcogenide Percentage (%)")
    if show_y_label:
        ax.set_ylabel("Uncertainty", color=COLORS[0])
    ax.tick_params(axis="y", labelcolor=COLORS[0])
    ax.set_xticks(np.arange(0.0, 1.1, 0.1))
    ax.set_xticklabels([int(x * 100) for x in np.arange(0.0, 1.1, 0.1)])

    ax2.errorbar(
        merged["non_target_frac"],
        merged[y_metric + "_x"],
        yerr=merged[yerr_metric + "_y"],
        fmt="s--",
        elinewidth=ELINEWIDTH,
        linewidth=LINEWIDTH,
        markersize=MARKERSIZE,
        markeredgewidth=MARKEREDGEWIDTH,
        capsize=CAPSIZE,
        capthick=CAPSTICK,
        color=COLORS[1],
        ecolor=ECOLORS[1],
        markerfacecolor=COLORS[1],
        markeredgecolor="white",
    )
    ax2.set_ylabel(ylabel, color=COLORS[1])
    ax2.tick_params(axis="y", labelcolor=COLORS[1])


all_per_metrics_df = pd.read_pickle("../figures/calcogenide_ood_all_per_metrics.pkl")
all_un_metrics_df = pd.read_pickle("../figures/calcogenide_ood_all_un_metrics.pkl")

unc_means = pd.concat(all_un_metrics_df).iloc[0::2, :]
unc_stds = pd.concat(all_un_metrics_df).iloc[1::2, :]
per_means = pd.concat(all_per_metrics_df).iloc[0::2, :]
per_stds = pd.concat(all_per_metrics_df).iloc[1::2, :]

unc = pd.merge(unc_means, unc_stds, on="non_target_frac")
per = pd.merge(per_means, per_stds, on="non_target_frac")
merged = pd.merge(unc, per, on="non_target_frac")

plt.rc("font", family="Helvetica Light", serif="Helvetica Light", size=20)
plt.rcParams["axes.linewidth"] = 1.5
plt.rcParams["xtick.major.size"] = 8
plt.rcParams["xtick.major.width"] = 1.5
plt.rcParams["ytick.major.size"] = 8
plt.rcParams["ytick.major.width"] = 1.5
plt.rcParams["ytick.direction"] = "in"
plt.rcParams["xtick.direction"] = "in"
plt.rcParams["legend.markerscale"] = 2
plt.rcParams["mathtext.it"] = "Helvetica Light:italic"
plt.rcParams["mathtext.rm"] = "Helvetica Light"
plt.rcParams["mathtext.default"] = "regular"

COLORS = ["#084c61", "#db504a"]
ECOLORS = ["#084c6180", "#db4f4a80"]
ELINEWIDTH = 1.5
LINEWIDTH = 2
MARKERSIZE = 10
CAPSIZE = 5
CAPSTICK = 2
MARKEREDGEWIDTH = 2

fig, axs = plt.subplots(2, 2, figsize=(18, 14), sharex=True, sharey=True)

plot_unc(
    axs[0, 0],
    axs[0, 0].twinx(),
    "Test_MAE",
    "Test_MAE",
    "MAE",
    show_legend=True,
    show_x_label=False,
    show_y_label=True,
)
plot_unc(
    axs[0, 1],
    axs[0, 1].twinx(),
    "Test_RMSE",
    "Test_RMSE",
    "RMSE",
    show_legend=False,
    show_x_label=False,
    show_y_label=False,
)
plot_unc(
    axs[1, 0],
    axs[1, 0].twinx(),
    "Test_MSE",
    "Test_MSE",
    "MSE",
    show_legend=False,
    show_x_label=True,
    show_y_label=True,
)
plot_unc(
    axs[1, 1],
    axs[1, 1].twinx(),
    "Test_R2",
    "Test_R2",
    r"$R^{2}$",
    show_legend=False,
    show_x_label=True,
    show_y_label=False,
)

plt.subplots_adjust(wspace=0.16, hspace=0.07)
plt.savefig(
    "../figures/errorbar_chalcogenide_uncertainty.png", dpi=300, bbox_inches="tight"
)
plt.show()

In [ ]:
def plot_unc(
    ax, ax2, y_metric, ylabel, show_legend=False, show_x_label=True, show_y_label=True
):

    ax.plot(
        merged["non_target_frac"],
        merged["Mean_Epistemic_Unc_x"],
        "o-",
        linewidth=LINEWIDTH,
        markersize=MARKERSIZE,
        markeredgewidth=MARKEREDGEWIDTH,
        color=COLORS[0],
        markerfacecolor=COLORS[0],
        markeredgecolor="white",
        label="Epistemic",
    )
    ax.plot(
        merged["non_target_frac"],
        merged["Mean_Aleatoric_Unc_x"],
        "s--",
        linewidth=LINEWIDTH,
        markersize=MARKERSIZE,
        markeredgewidth=MARKEREDGEWIDTH,
        color=COLORS[0],
        markerfacecolor=COLORS[0],
        markeredgecolor="white",
        label="Aleatoric",
    )
    ax.plot(
        merged["non_target_frac"],
        merged["Mean_Total_Unc_x"],
        "^-",
        linewidth=LINEWIDTH,
        markersize=MARKERSIZE,
        markeredgewidth=MARKEREDGEWIDTH,
        color=COLORS[0],
        markerfacecolor=COLORS[0],
        markeredgecolor="white",
        label="Total",
    )

    if show_legend:
        ax.legend(frameon=False, loc="upper right")

    if show_x_label:
        ax.set_xlabel("Chalcogenide Percentage (%)")
    if show_y_label:
        ax.set_ylabel("Uncertainty", color=COLORS[0])

    ax.tick_params(axis="y", labelcolor=COLORS[0])
    ax.set_xticks(np.arange(0.0, 1.1, 0.1))
    ax.set_xticklabels([int(x * 100) for x in np.arange(0.0, 1.1, 0.1)])

    ax2.plot(
        merged["non_target_frac"],
        merged[y_metric + "_x"],
        "s--",
        linewidth=LINEWIDTH,
        markersize=MARKERSIZE,
        markeredgewidth=MARKEREDGEWIDTH,
        color=COLORS[1],
        markerfacecolor=COLORS[1],
        markeredgecolor="white",
    )

    ax2.set_ylabel(ylabel, color=COLORS[1])
    ax2.tick_params(axis="y", labelcolor=COLORS[1])


all_per_metrics_df = pd.read_pickle("../figures/calcogenide_ood_all_per_metrics.pkl")
all_un_metrics_df = pd.read_pickle("../figures/calcogenide_ood_all_un_metrics.pkl")

unc_means = pd.concat(all_un_metrics_df).iloc[0::2, :]
unc_stds = pd.concat(all_un_metrics_df).iloc[1::2, :]
per_means = pd.concat(all_per_metrics_df).iloc[0::2, :]
per_stds = pd.concat(all_per_metrics_df).iloc[1::2, :]

unc = pd.merge(unc_means, unc_stds, on="non_target_frac")
per = pd.merge(per_means, per_stds, on="non_target_frac")
merged = pd.merge(unc, per, on="non_target_frac")

plt.rc("font", family="Helvetica Light", serif="Helvetica Light", size=20)
plt.rcParams["axes.linewidth"] = 1.5
plt.rcParams["xtick.major.size"] = 8
plt.rcParams["xtick.major.width"] = 1.5
plt.rcParams["ytick.major.size"] = 8
plt.rcParams["ytick.major.width"] = 1.5
plt.rcParams["ytick.direction"] = "in"
plt.rcParams["xtick.direction"] = "in"
plt.rcParams["legend.markerscale"] = 2
plt.rcParams["mathtext.it"] = "Helvetica Light:italic"
plt.rcParams["mathtext.rm"] = "Helvetica Light"
plt.rcParams["mathtext.default"] = "regular"

COLORS = ["#084c61", "#db504a"]
ECOLORS = ["#084c6180", "#db4f4a80"]
ELINEWIDTH = 1.5
LINEWIDTH = 2
MARKERSIZE = 10
CAPSIZE = 5
CAPSTICK = 2
MARKEREDGEWIDTH = 2

fig, axs = plt.subplots(2, 2, figsize=(18, 14), sharex=True, sharey=True)

plot_unc(
    axs[0, 0],
    axs[0, 0].twinx(),
    "Test_MAE",
    "MAE",
    show_legend=True,
    show_x_label=False,
    show_y_label=True,
)

plot_unc(
    axs[0, 1],
    axs[0, 1].twinx(),
    "Test_RMSE",
    "RMSE",
    show_legend=False,
    show_x_label=False,
    show_y_label=False,
)

plot_unc(
    axs[1, 0],
    axs[1, 0].twinx(),
    "Test_MSE",
    "MSE",
    show_legend=False,
    show_x_label=True,
    show_y_label=True,
)

plot_unc(
    axs[1, 1],
    axs[1, 1].twinx(),
    "Test_R2",
    r"$R^{2}$",
    show_legend=False,
    show_x_label=True,
    show_y_label=False,
)

plt.subplots_adjust(wspace=0.16, hspace=0.07)
plt.savefig(
    "../figures/no_errorbar_chalcogenide_uncertainty.png", dpi=300, bbox_inches="tight"
)
plt.show()

In [ ]:
all_results = []
root_dir = "./uncertainty_chalcogenide_results/not_target_frac_1.0"
for split in sorted(os.listdir(root_dir)):
    split_path = os.path.join(root_dir, split)

    if os.path.isdir(split_path) and split.startswith("run_"):

        for model in sorted(os.listdir(split_path)):
            model_path = os.path.join(split_path, model)

            if os.path.isdir(model_path) and model.startswith("model_"):

                test_file = os.path.join(model_path, "results_enn_test_set.csv")

                if os.path.exists(test_file):
                    df = pd.read_csv(test_file)
                    all_results.append(df)

In [ ]:
results = pd.DataFrame(
    np.mean(all_results, axis=0),
    columns=["y_test", "y_pred", "aleatoric", "epistemic", "total"],
)

In [ ]:
results.to_csv(
    "../figures/scatter_plot_residual_error_total_unc_calcogenide.csv", index=False
)

In [ ]:
results = pd.DataFrame(
    np.mean(all_results, axis=0),
    columns=["y_test", "y_pred", "aleatoric", "epistemic", "total"],
)

results = pd.read_csv(
    "../figures/scatter_plot_residual_error_total_unc_calcogenide.csv"
)

results["residual"] = abs(results["y_test"] - results["y_pred"])
results["nor_epist"] = results["epistemic"] / results["total"]
results["nor_aleat"] = results["aleatoric"] / results["total"]

In [ ]:
plt.rc("font", family="Helvetica Light", serif="Helvetica Light", size=20)
plt.rcParams["axes.linewidth"] = 1.5
plt.rcParams["xtick.major.size"] = 8
plt.rcParams["xtick.major.width"] = 1.5
plt.rcParams["ytick.major.size"] = 8
plt.rcParams["ytick.major.width"] = 1.5
plt.rcParams["ytick.direction"] = "in"
plt.rcParams["xtick.direction"] = "in"
plt.rcParams["legend.markerscale"] = 2
plt.rcParams["mathtext.it"] = "Helvetica Light:italic"
plt.rcParams["mathtext.rm"] = "Helvetica Light"
plt.rcParams["mathtext.default"] = "regular"

cmap_custom = LinearSegmentedColormap.from_list("custom", ["#db504a80", "#084c6180"])

plt.figure(figsize=(10, 8))
scatter = plt.scatter(
    results["total"],
    results["residual"],
    c=results["nor_epist"],
    cmap=cmap_custom,
    s=60,
    alpha=0.7,
    edgecolors="black",
    linewidths=1.4,
    vmin=0.0,
    vmax=1.0,
)

plt.yticks([0.0, 1.0, 2.0, 3.0])

cbar = plt.colorbar(scatter)
cbar.set_label("Uncertainty", labelpad=-80)

cbar.set_ticks([0.0, 1.0])

cbar.set_ticklabels(["Aleatoric", "Epistemic"])

plt.ylabel("Total Uncertainty")
plt.xlabel("Residual Error")
plt.xlim(-0.2, 9.2)
plt.ylim(-0.1, 3.2)
plt.savefig("../figures/chalcogenide_scatter_plot.png", dpi=300, bbox_inches="tight")
plt.show()

## 6-2- T-based


### 6-2-1- Codes

In [ ]:
def create_strata(df, n_bins=5):
    """Creates a combined stratification column based on kL and T quantiles."""

    df[f"kL_bins"] = pd.qcut(df["kL"], q=n_bins, labels=False, duplicates="drop")
    df[f"T_bins"] = pd.qcut(df["T"], q=n_bins, labels=False, duplicates="drop")

    df["Strata"] = df["T_bins"].astype(str) + "_" + df["kL_bins"].astype(str)
    return df

In [ ]:
def run_uncertainty_experiment(
    df_all: pd.DataFrame,
    col_name: str,
    runs: int,
    output_dir: Union[str, Path],
    not_target_frac: float = 0,
) -> None:
    """
    Run uncertainty experiments across multiple random seeds and model instances.

    Parameters
    ----------
    df_all : pandas.DataFrame
        Input dataset containing the target column `col_name`.
    col_name : str
        Boolean column name indicating the target class (e.g., 'T_high').
    runs : int
        Number of independent experiment runs.
    output_dir : str or pathlib.Path
        Directory path for saving outputs, model checkpoints, and split files.
    not_target_frac : float, optional
        Fraction of non-target samples to include in the training set (default=0).
    """
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    output_dir = Path(output_dir)

    df_high = df_all[df_all[col_name] == True].copy()
    df_non_high = df_all[df_all[col_name] == False].copy()

    df_high = create_strata(df_high)
    df_non_high = create_strata(df_non_high)

    df_non_high_rem, val_non_high_component = train_test_split(
        df_non_high,
        test_size=200,
        stratify=df_non_high["Strata"],
        random_state=42,
    )

    base_train_set = df_non_high_rem

    high_ood_pool, df_high_tv = train_test_split(
        df_high,
        test_size=600,
        stratify=df_high["Strata"],
        random_state=42,
    )

    test_set, val_high_component = train_test_split(
        df_high_tv, test_size=200, stratify=df_high_tv["Strata"], random_state=42
    )

    non_target_pool = high_ood_pool

    final_validation_set = pd.concat([val_non_high_component, val_high_component])

    val_set = (
        final_validation_set.drop(columns=["T_bins", "kL_bins", "Strata", col_name])
        .sample(frac=1, random_state=42)
        .reset_index(drop=True)
    )
    test_set = (
        test_set.drop(columns=["T_bins", "kL_bins", "Strata", col_name])
        .sample(frac=1, random_state=42)
        .reset_index(drop=True)
    )

    base_train_set = (
        base_train_set.drop(columns=["T_bins", "kL_bins", "Strata", col_name])
        .sample(frac=1, random_state=42)
        .reset_index(drop=True)
    )

    non_target_pool = (
        non_target_pool.drop(columns=["T_bins", "kL_bins", "Strata", col_name])
        .sample(frac=1, random_state=42)
        .reset_index(drop=True)
    )

    for run_idx in range(1, runs + 1):
        print(f"\nRun: {run_idx}")
        random_seed = 42 + run_idx

        run_dir = output_dir / f"run_{run_idx}"
        run_dir.mkdir(parents=True, exist_ok=True)

        if not_target_frac:
            extra_nontarget = non_target_pool.sample(
                frac=not_target_frac, random_state=random_seed
            )
            train_set = pd.concat([base_train_set, extra_nontarget], axis=0)

        else:
            train_set = base_train_set

        print("Train shape:", train_set.shape)

        X_val = val_set.iloc[:, 1:-2].to_numpy()
        X_train = train_set.iloc[:, 1:-2].to_numpy()
        X_test = test_set.iloc[:, 1:-2].to_numpy()
        y_val = val_set.iloc[:, -2].to_numpy()
        y_train = train_set.iloc[:, -2].to_numpy()
        y_test = test_set.iloc[:, -2].to_numpy()

        split_dict = {
            "X_train": X_train,
            "X_val": X_val,
            "X_test": X_test,
            "y_train": y_train,
            "y_val": y_val,
            "y_test": y_test,
        }

        with open(run_dir / "dataset_split.pkl", "wb") as f:
            pickle.dump(split_dict, f, protocol=pickle.HIGHEST_PROTOCOL)

        in_features = X_train.shape[1]
        print(f"Input features for model: {in_features}")

        train_loader, val_loader, test_loader = create_dataloaders(
            X_train,
            X_val,
            X_test,
            y_train,
            y_val,
            y_test,
            batch_size=CONFIG["batch_size"],
            shuffle_train=CONFIG["shuffle_train"],
        )

        dataloaders = {"train": train_loader, "val": val_loader, "test": test_loader}

        for model_idx in range(1, 6):
            print(f"\n--- Training model {model_idx} on Run {run_idx} ---")
            set_seed(42 * (run_idx * 100) + model_idx)

            model = FFNN(
                in_features,
                CONFIG["model_params"]["n_layers"],
                CONFIG["model_params"]["act_fn"],
                CONFIG["model_params"]["num_neu_list"],
                CONFIG["model_params"]["p"],
            )
            model.apply(lambda m: init_weights(m, nonlinearity=model.act_fn_name))
            model = model.to(device)

            optimizer = getattr(
                torch.optim, CONFIG["training_params"]["optimizer_name"]
            )(
                model.parameters(),
                lr=CONFIG["training_params"]["learning_rate"],
                weight_decay=CONFIG["training_params"]["weight_decay"],
            )
            scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
                optimizer,
                mode="min",
                factor=0.5,
                patience=CONFIG["training_params"]["scheduler_patience"],
            )

            early_stopper = EarlyStopper(
                patience=CONFIG["training_params"]["early_stopping_patience"],
                delta=CONFIG["training_params"]["early_stopping_delta"],
                minimize=True,
            )

            model_dir = run_dir / f"model_{model_idx}"
            model_dir.mkdir(parents=True, exist_ok=True)

            best_val_rmse, best_model_path = train_and_val_proc(
                model,
                optimizer,
                evidential_regression,
                CONFIG["training_params"]["epochs"],
                scheduler,
                dataloaders,
                early_stopper,
                device,
                model_dir,
                lamb_loss=CONFIG["training_params"]["loss_lamb"],
                target_inverse_transform=(
                    CONFIG["target_transform"]["inverse_function_for_metrics"]
                    if CONFIG["target_transform"]["enabled"]
                    else None
                ),
            )

            print(f"Model {model_idx} best validation RMSE: {best_val_rmse:.3f}")

            final_model = FFNN(
                in_features,
                CONFIG["model_params"]["n_layers"],
                CONFIG["model_params"]["act_fn"],
                CONFIG["model_params"]["num_neu_list"],
                CONFIG["model_params"]["p"],
            )
            final_model.load_state_dict(
                torch.load(best_model_path, map_location=device)
            )
            final_model.to(device)

            test_proc(
                final_model,
                nig_nll,
                dataloaders,
                device,
                model_dir,
                target_inverse_transform=(
                    CONFIG["target_transform"]["inverse_function_for_metrics"]
                    if CONFIG["target_transform"]["enabled"]
                    else None
                ),
            )

            val_proc(
                final_model,
                nig_nll,
                dataloaders,
                device,
                model_dir,
                target_inverse_transform=(
                    CONFIG["target_transform"]["inverse_function_for_metrics"]
                    if CONFIG["target_transform"]["enabled"]
                    else None
                ),
            )

In [ ]:
def process_uncertainty_results(root_dir: Union[str, Path]) -> None:
    """
    Process and aggregate uncertainty experiment results stored under a root directory.

    Parameters
    ----------
    root_dir : str or pathlib.Path
        Path to the root directory containing 'run_*' subdirectories.

    Notes
    -----
    This function performs three sequential tasks:
        1. Aggregates all 'results_ffnn_model_test_set.csv' files across runs.
        2. Computes calibrated uncertainties using 'results_enn_test_set.csv' and 'results_enn_val_set.csv'.
        3. Evaluates and aggregates uncertainty metrics from calibrated results.
    """

    root_dir = Path(root_dir)

    # Step 1: Aggregate raw FFNN test results
    all_results = []
    for split in sorted(os.listdir(root_dir)):
        split_path = root_dir / split

        if split_path.is_dir() and split.startswith("run_"):

            for model in sorted(os.listdir(split_path)):
                model_path = split_path / model
                if model_path.is_dir() and model.startswith("model_"):

                    test_file = model_path / "results_ffnn_model_test_set.csv"
                    if test_file.exists():
                        df = pd.read_csv(test_file)
                        df["split"] = split
                        df["model"] = model
                        all_results.append(df)

    if all_results:
        aggregated_results = pd.concat(all_results, ignore_index=True)
        aggregated_results.to_csv(root_dir / "aggregated_test_results.csv", index=False)
        print("Aggregated results saved to 'aggregated_test_results.csv'")
    else:
        print("No FFNN test results found!")

    # Step 2: Evaluate and aggregate calibrated uncertainty metrics
    all_metrics = []
    for split in sorted(os.listdir(root_dir)):

        split_path = root_dir / split

        if split_path.is_dir() and split.startswith("run_"):

            for model in sorted(os.listdir(split_path)):
                model_path = split_path / model

                if model_path.is_dir() and model.startswith("model_"):

                    test_file = model_path / "results_enn_test_set.csv"

                    if test_file.exists():

                        unc_df = pd.read_csv(test_file)
                        total_metrics = evaluate_uncertainty(
                            y_true=unc_df["y_test"].to_numpy(),
                            y_pred=unc_df["y_pred"].to_numpy(),
                            total_unc=unc_df["total_unc"].to_numpy(),
                        )

                        epistemic_metrics = evaluate_uncertainty(
                            y_true=unc_df["y_test"].to_numpy(),
                            y_pred=unc_df["y_pred"].to_numpy(),
                            total_unc=unc_df["epistemic_unc"].to_numpy(),
                        )

                        aleatoric_metrics = evaluate_uncertainty(
                            y_true=unc_df["y_test"].to_numpy(),
                            y_pred=unc_df["y_pred"].to_numpy(),
                            total_unc=unc_df["aleatoric_unc"].to_numpy(),
                        )

                        all_metrics.append(
                            {
                                "split": split,
                                "model": model,
                                "Total_NLL": total_metrics["NLL"],
                                "Total_MCA": total_metrics["MiscalibrationArea"],
                                "Total_SpearmanR": total_metrics["SpearmanR"],
                                "Epistemic_NLL": epistemic_metrics["NLL"],
                                "Epistemic_MCA": epistemic_metrics[
                                    "MiscalibrationArea"
                                ],
                                "Epistemic_SpearmanR": epistemic_metrics["SpearmanR"],
                                "Aleatoric_NLL": aleatoric_metrics["NLL"],
                                "Aleatoric_MCA": aleatoric_metrics[
                                    "MiscalibrationArea"
                                ],
                                "Aleatoric_SpearmanR": aleatoric_metrics["SpearmanR"],
                                "Mean_Total_Unc": unc_df["total_unc"].median(),
                                "Mean_Epistemic_Unc": unc_df["epistemic_unc"].median(),
                                "Mean_Aleatoric_Unc": unc_df["aleatoric_unc"].median(),
                            }
                        )

    if all_metrics:

        df_metrics = pd.DataFrame(all_metrics)
        output_path = root_dir / "unc_aggregated_test_metrics.csv"
        df_metrics.to_csv(output_path, index=False)
        print(f"Aggregated results saved to '{output_path.name}'")

    else:

        print("No calibrated uncertainty metrics found!")

### 6-2-2- Train: Low, Test: High

In [ ]:
df_all = pd.read_parquet("../feature_vectors/total_dataset.parquet")

df_all.loc[df_all["T"] >= 600.0, "T_high"] = True
df_all.loc[df_all["T"] < 600.0, "T_high"] = False

In [ ]:
for frac in [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]:
    run_uncertainty_experiment(
        df_all,
        "T_high",
        10,
        f"./uncertainty_T_high_results/not_target_frac_{frac}",
        not_target_frac=frac,
    )

In [ ]:
for frac in [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]:
    process_uncertainty_results(f"./uncertainty_T_high_results/not_target_frac_{frac}")

In [ ]:
all_per_metrics_df = []
all_un_metrics_df = []

for frac in [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]:

    per_metrics_path = f"./uncertainty_T_high_results/not_target_frac_{frac}/aggregated_test_results.csv"
    un_metrics_path = f"./uncertainty_T_high_results/not_target_frac_{frac}/unc_aggregated_test_metrics.csv"

    per_metrics_df = pd.read_csv(per_metrics_path).describe().iloc[1:3, :]
    un_metrics_df = pd.read_csv(un_metrics_path).describe().iloc[1:3, :]

    per_metrics_df["non_target_frac"] = frac
    un_metrics_df["non_target_frac"] = frac

    all_per_metrics_df.append(per_metrics_df)
    all_un_metrics_df.append(un_metrics_df)

In [ ]:
pd.to_pickle(all_per_metrics_df, "../figures/t_high_ood_all_per_metrics.pkl")
pd.to_pickle(all_un_metrics_df, "../figures/t_high_ood_all_un_metrics.pkl")

In [ ]:
def plot_unc(
    ax,
    ax2,
    y_metric,
    yerr_metric,
    ylabel,
    show_legend=False,
    show_x_label=True,
    show_y_label=True,
):
    ax.errorbar(
        merged["non_target_frac"],
        merged["Mean_Epistemic_Unc_x"],
        yerr=merged["Mean_Epistemic_Unc_y"],
        fmt="o-",
        elinewidth=ELINEWIDTH,
        linewidth=LINEWIDTH,
        markersize=MARKERSIZE,
        markeredgewidth=MARKEREDGEWIDTH,
        capsize=CAPSIZE,
        capthick=CAPSTICK,
        color=COLORS[0],
        ecolor=ECOLORS[0],
        markerfacecolor=COLORS[0],
        markeredgecolor="white",
        label="Epistemic",
    )
    ax.errorbar(
        merged["non_target_frac"],
        merged["Mean_Aleatoric_Unc_x"],
        yerr=merged["Mean_Aleatoric_Unc_y"],
        fmt="s--",
        elinewidth=ELINEWIDTH,
        linewidth=LINEWIDTH,
        markersize=MARKERSIZE,
        markeredgewidth=MARKEREDGEWIDTH,
        capsize=CAPSIZE,
        capthick=CAPSTICK,
        color=COLORS[0],
        ecolor=ECOLORS[0],
        markerfacecolor=COLORS[0],
        markeredgecolor="white",
        label="Aleatoric",
    )
    ax.errorbar(
        merged["non_target_frac"],
        merged["Mean_Total_Unc_x"],
        yerr=merged["Mean_Total_Unc_y"],
        fmt="^-",
        elinewidth=ELINEWIDTH,
        linewidth=LINEWIDTH,
        markersize=MARKERSIZE,
        markeredgewidth=MARKEREDGEWIDTH,
        capsize=CAPSIZE,
        capthick=CAPSTICK,
        color=COLORS[0],
        ecolor=ECOLORS[0],
        markerfacecolor=COLORS[0],
        markeredgecolor="white",
        label="Total",
    )

    if show_legend:
        ax.legend(frameon=False, loc="upper right")
    if show_x_label:
        ax.set_xlabel("High T Material Percentage (%)")
    if show_y_label:
        ax.set_ylabel("Uncertainty", color=COLORS[0])
    ax.tick_params(axis="y", labelcolor=COLORS[0])
    ax.set_xticks(np.arange(0.0, 1.1, 0.1))
    ax.set_xticklabels([int(x * 100) for x in np.arange(0.0, 1.1, 0.1)])

    ax2.errorbar(
        merged["non_target_frac"],
        merged[y_metric + "_x"],
        yerr=merged[yerr_metric + "_y"],
        fmt="s--",
        elinewidth=ELINEWIDTH,
        linewidth=LINEWIDTH,
        markersize=MARKERSIZE,
        markeredgewidth=MARKEREDGEWIDTH,
        capsize=CAPSIZE,
        capthick=CAPSTICK,
        color=COLORS[1],
        ecolor=ECOLORS[1],
        markerfacecolor=COLORS[1],
        markeredgecolor="white",
    )
    ax2.set_ylabel(ylabel, color=COLORS[1])
    ax2.tick_params(axis="y", labelcolor=COLORS[1])


all_per_metrics_df = pd.read_pickle("../figures/t_high_ood_all_per_metrics.pkl")
all_un_metrics_df = pd.read_pickle("../figures/t_high_ood_all_un_metrics.pkl")

unc_means = pd.concat(all_un_metrics_df).iloc[0::2, :]
unc_stds = pd.concat(all_un_metrics_df).iloc[1::2, :]
per_means = pd.concat(all_per_metrics_df).iloc[0::2, :]
per_stds = pd.concat(all_per_metrics_df).iloc[1::2, :]

unc = pd.merge(unc_means, unc_stds, on="non_target_frac")
per = pd.merge(per_means, per_stds, on="non_target_frac")
merged = pd.merge(unc, per, on="non_target_frac")

plt.rc("font", family="Helvetica Light", serif="Helvetica Light", size=20)
plt.rcParams["axes.linewidth"] = 1.5
plt.rcParams["xtick.major.size"] = 8
plt.rcParams["xtick.major.width"] = 1.5
plt.rcParams["ytick.major.size"] = 8
plt.rcParams["ytick.major.width"] = 1.5
plt.rcParams["ytick.direction"] = "in"
plt.rcParams["xtick.direction"] = "in"
plt.rcParams["legend.markerscale"] = 2
plt.rcParams["mathtext.it"] = "Helvetica Light:italic"
plt.rcParams["mathtext.rm"] = "Helvetica Light"
plt.rcParams["mathtext.default"] = "regular"

COLORS = ["#084c61", "#db504a"]
ECOLORS = ["#084c61a0", "#db4f4aa2"]
ELINEWIDTH = 1.5
LINEWIDTH = 2
MARKERSIZE = 10
CAPSIZE = 5
CAPSTICK = 2
MARKEREDGEWIDTH = 2

fig, axs = plt.subplots(2, 2, figsize=(18, 14), sharex=True, sharey=True)

plot_unc(
    axs[0, 0],
    axs[0, 0].twinx(),
    "Test_MAE",
    "Test_MAE",
    "MAE",
    show_legend=True,
    show_x_label=False,
    show_y_label=True,
)
plot_unc(
    axs[0, 1],
    axs[0, 1].twinx(),
    "Test_RMSE",
    "Test_RMSE",
    "RMSE",
    show_legend=False,
    show_x_label=False,
    show_y_label=False,
)
plot_unc(
    axs[1, 0],
    axs[1, 0].twinx(),
    "Test_MSE",
    "Test_MSE",
    "MSE",
    show_legend=False,
    show_x_label=True,
    show_y_label=True,
)
plot_unc(
    axs[1, 1],
    axs[1, 1].twinx(),
    "Test_R2",
    "Test_R2",
    r"$R^{2}$",
    show_legend=False,
    show_x_label=True,
    show_y_label=False,
)

plt.subplots_adjust(wspace=0.16, hspace=0.07)
plt.savefig("../figures/errorbar_t_high_uncertainty.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
def plot_unc(
    ax, ax2, y_metric, ylabel, show_legend=False, show_x_label=True, show_y_label=True
):

    ax.plot(
        merged["non_target_frac"],
        merged["Mean_Epistemic_Unc_x"],
        "o-",
        linewidth=LINEWIDTH,
        markersize=MARKERSIZE,
        markeredgewidth=MARKEREDGEWIDTH,
        color=COLORS[0],
        markerfacecolor=COLORS[0],
        markeredgecolor="white",
        label="Epistemic",
    )
    ax.plot(
        merged["non_target_frac"],
        merged["Mean_Aleatoric_Unc_x"],
        "s--",
        linewidth=LINEWIDTH,
        markersize=MARKERSIZE,
        markeredgewidth=MARKEREDGEWIDTH,
        color=COLORS[0],
        markerfacecolor=COLORS[0],
        markeredgecolor="white",
        label="Aleatoric",
    )
    ax.plot(
        merged["non_target_frac"],
        merged["Mean_Total_Unc_x"],
        "^-",
        linewidth=LINEWIDTH,
        markersize=MARKERSIZE,
        markeredgewidth=MARKEREDGEWIDTH,
        color=COLORS[0],
        markerfacecolor=COLORS[0],
        markeredgecolor="white",
        label="Total",
    )

    if show_legend:
        ax.legend(frameon=False, loc="upper right")

    if show_x_label:
        ax.set_xlabel("High T Material (%)")
    if show_y_label:
        ax.set_ylabel("Uncertainty", color=COLORS[0])

    ax.tick_params(axis="y", labelcolor=COLORS[0])
    ax.set_xticks(np.arange(0.0, 1.1, 0.1))
    ax.set_xticklabels([int(x * 100) for x in np.arange(0.0, 1.1, 0.1)])

    ax2.plot(
        merged["non_target_frac"],
        merged[y_metric + "_x"],
        "s--",
        linewidth=LINEWIDTH,
        markersize=MARKERSIZE,
        markeredgewidth=MARKEREDGEWIDTH,
        color=COLORS[1],
        markerfacecolor=COLORS[1],
        markeredgecolor="white",
    )

    ax2.set_ylabel(ylabel, color=COLORS[1])
    ax2.tick_params(axis="y", labelcolor=COLORS[1])


unc_means = pd.concat(all_un_metrics_df).iloc[0::2, :]
unc_stds = pd.concat(all_un_metrics_df).iloc[1::2, :]
per_means = pd.concat(all_per_metrics_df).iloc[0::2, :]
per_stds = pd.concat(all_per_metrics_df).iloc[1::2, :]

unc = pd.merge(unc_means, unc_stds, on="non_target_frac")
per = pd.merge(per_means, per_stds, on="non_target_frac")
merged = pd.merge(unc, per, on="non_target_frac")

plt.rc("font", family="Helvetica Light", serif="Helvetica Light", size=20)
plt.rcParams["axes.linewidth"] = 1.5
plt.rcParams["xtick.major.size"] = 8
plt.rcParams["xtick.major.width"] = 1.5
plt.rcParams["ytick.major.size"] = 8
plt.rcParams["ytick.major.width"] = 1.5
plt.rcParams["ytick.direction"] = "in"
plt.rcParams["xtick.direction"] = "in"
plt.rcParams["legend.markerscale"] = 2
plt.rcParams["mathtext.it"] = "Helvetica Light:italic"
plt.rcParams["mathtext.rm"] = "Helvetica Light"
plt.rcParams["mathtext.default"] = "regular"

COLORS = ["#084c61", "#db504a"]
ECOLORS = ["#084c61a0", "#db4f4aa2"]
ELINEWIDTH = 1.5
LINEWIDTH = 2
MARKERSIZE = 10
CAPSIZE = 5
CAPSTICK = 2
MARKEREDGEWIDTH = 2

fig, axs = plt.subplots(2, 2, figsize=(18, 14), sharex=True, sharey=True)

plot_unc(
    axs[0, 0],
    axs[0, 0].twinx(),
    "Test_MAE",
    "MAE",
    show_legend=True,
    show_x_label=False,
    show_y_label=True,
)

plot_unc(
    axs[0, 1],
    axs[0, 1].twinx(),
    "Test_RMSE",
    "RMSE",
    show_legend=False,
    show_x_label=False,
    show_y_label=False,
)

plot_unc(
    axs[1, 0],
    axs[1, 0].twinx(),
    "Test_MSE",
    "MSE",
    show_legend=False,
    show_x_label=True,
    show_y_label=True,
)

plot_unc(
    axs[1, 1],
    axs[1, 1].twinx(),
    "Test_R2",
    r"$R^{2}$",
    show_legend=False,
    show_x_label=True,
    show_y_label=False,
)

plt.subplots_adjust(wspace=0.16, hspace=0.07)
plt.savefig("no_errorbar_t_high_uncertainty.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
all_results = []
root_dir = "./uncertainty_T_high_results/not_target_frac_1.0"
for split in sorted(os.listdir(root_dir)):
    split_path = os.path.join(root_dir, split)

    if os.path.isdir(split_path) and split.startswith("run_"):

        for model in sorted(os.listdir(split_path)):
            model_path = os.path.join(split_path, model)

            if os.path.isdir(model_path) and model.startswith("model_"):

                test_file = os.path.join(model_path, "results_enn_test_set.csv")

                if os.path.exists(test_file):
                    df = pd.read_csv(test_file)
                    all_results.append(df)

In [ ]:
results = pd.DataFrame(
    np.mean(all_results, axis=0),
    columns=["y_test", "y_pred", "aleatoric", "epistemic", "total"],
)

results.to_csv(
    "../figures/scatter_plot_residual_error_total_unc_high_t.csv", index=False
)

results["residual"] = abs(results["y_test"] - results["y_pred"])
results["nor_epist"] = results["epistemic"] / results["total"]
results["nor_aleat"] = results["aleatoric"] / results["total"]

In [ ]:
plt.rc("font", family="Helvetica Light", serif="Helvetica Light", size=20)
plt.rcParams["axes.linewidth"] = 1.5
plt.rcParams["xtick.major.size"] = 8
plt.rcParams["xtick.major.width"] = 1.5
plt.rcParams["ytick.major.size"] = 8
plt.rcParams["ytick.major.width"] = 1.5
plt.rcParams["ytick.direction"] = "in"
plt.rcParams["xtick.direction"] = "in"
plt.rcParams["legend.markerscale"] = 2
plt.rcParams["mathtext.it"] = "Helvetica Light:italic"
plt.rcParams["mathtext.rm"] = "Helvetica Light"
plt.rcParams["mathtext.default"] = "regular"

cmap_custom = LinearSegmentedColormap.from_list("custom", ["#db504a80", "#084c6180"])

plt.figure(figsize=(10, 8))
scatter = plt.scatter(
    results["total"],
    results["residual"],
    c=results["nor_epist"],
    cmap=cmap_custom,
    s=60,
    alpha=0.7,
    edgecolors="black",
    linewidths=1.4,
    vmin=0.0,
    vmax=1.0,
)

cbar = plt.colorbar(scatter)
cbar.set_label("Uncertainty", labelpad=-80)

cbar.set_ticks([0.0, 1.0])

cbar.set_ticklabels(["Aleatoric", "Epistemic"])

plt.ylabel("Total Uncertainty")
plt.xlabel("Residual Error")
plt.xlim(-0.2, 6)
plt.ylim(-0.1, 4.5)
plt.savefig("../figures/t_high_parity_plot.png", dpi=300, bbox_inches="tight")
plt.show()

### 6-2-3- Train: high, Test: low

In [ ]:
df_all = pd.read_parquet("../feature_vectors/total_dataset.parquet")

df_all.loc[df_all["T"] >= 600.0, "T_low"] = False
df_all.loc[df_all["T"] < 600.0, "T_low"] = True

In [ ]:
for frac in [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]:
    run_uncertainty_experiment(
        df_all,
        "T_low",
        10,
        f"./uncertainty_T_low_results/not_target_frac_{frac}",
        not_target_frac=frac,
    )

In [ ]:
for frac in [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]:
    process_uncertainty_results(f"./uncertainty_T_low_results/not_target_frac_{frac}")

In [ ]:
all_per_metrics_df = []
all_un_metrics_df = []

for frac in [0.0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]:

    per_metrics_path = f"./uncertainty_T_low_results/not_target_frac_{frac}/aggregated_test_results.csv"
    un_metrics_path = f"./uncertainty_T_low_results/not_target_frac_{frac}/unc_aggregated_test_metrics.csv"

    per_metrics_df = pd.read_csv(per_metrics_path).describe().iloc[1:3, :]
    un_metrics_df = pd.read_csv(un_metrics_path).describe().iloc[1:3, :]

    per_metrics_df["non_target_frac"] = frac
    un_metrics_df["non_target_frac"] = frac

    all_per_metrics_df.append(per_metrics_df)
    all_un_metrics_df.append(un_metrics_df)

In [ ]:
pd.to_pickle(all_per_metrics_df, "../figures/t_low_ood_all_per_metrics.pkl")
pd.to_pickle(all_un_metrics_df, "../figures/t_low_ood_all_un_metrics.pkl")

In [ ]:
def plot_unc(
    ax,
    ax2,
    y_metric,
    yerr_metric,
    ylabel,
    show_legend=False,
    show_x_label=True,
    show_y_label=True,
):
    ax.errorbar(
        merged["non_target_frac"],
        merged["Mean_Epistemic_Unc_x"],
        yerr=merged["Mean_Epistemic_Unc_y"],
        fmt="o-",
        elinewidth=ELINEWIDTH,
        linewidth=LINEWIDTH,
        markersize=MARKERSIZE,
        markeredgewidth=MARKEREDGEWIDTH,
        capsize=CAPSIZE,
        capthick=CAPSTICK,
        color=COLORS[0],
        ecolor=ECOLORS[0],
        markerfacecolor=COLORS[0],
        markeredgecolor="white",
        label="Epistemic",
    )
    ax.errorbar(
        merged["non_target_frac"],
        merged["Mean_Aleatoric_Unc_x"],
        yerr=merged["Mean_Aleatoric_Unc_y"],
        fmt="s--",
        elinewidth=ELINEWIDTH,
        linewidth=LINEWIDTH,
        markersize=MARKERSIZE,
        markeredgewidth=MARKEREDGEWIDTH,
        capsize=CAPSIZE,
        capthick=CAPSTICK,
        color=COLORS[0],
        ecolor=ECOLORS[0],
        markerfacecolor=COLORS[0],
        markeredgecolor="white",
        label="Aleatoric",
    )
    ax.errorbar(
        merged["non_target_frac"],
        merged["Mean_Total_Unc_x"],
        yerr=merged["Mean_Total_Unc_y"],
        fmt="^-",
        elinewidth=ELINEWIDTH,
        linewidth=LINEWIDTH,
        markersize=MARKERSIZE,
        markeredgewidth=MARKEREDGEWIDTH,
        capsize=CAPSIZE,
        capthick=CAPSTICK,
        color=COLORS[0],
        ecolor=ECOLORS[0],
        markerfacecolor=COLORS[0],
        markeredgecolor="white",
        label="Total",
    )

    if show_legend:
        ax.legend(frameon=False, loc="upper right")
    if show_x_label:
        ax.set_xlabel("Low T Material Percentage (%)")
    if show_y_label:
        ax.set_ylabel("Uncertainty", color=COLORS[0])
    ax.tick_params(axis="y", labelcolor=COLORS[0])
    ax.set_xticks(np.arange(0.0, 1.1, 0.1))
    ax.set_xticklabels([int(x * 100) for x in np.arange(0.0, 1.1, 0.1)])

    ax2.errorbar(
        merged["non_target_frac"],
        merged[y_metric + "_x"],
        yerr=merged[yerr_metric + "_y"],
        fmt="s--",
        elinewidth=ELINEWIDTH,
        linewidth=LINEWIDTH,
        markersize=MARKERSIZE,
        markeredgewidth=MARKEREDGEWIDTH,
        capsize=CAPSIZE,
        capthick=CAPSTICK,
        color=COLORS[1],
        ecolor=ECOLORS[1],
        markerfacecolor=COLORS[1],
        markeredgecolor="white",
    )
    ax2.set_ylabel(ylabel, color=COLORS[1])
    ax2.tick_params(axis="y", labelcolor=COLORS[1])


all_per_metrics_df = pd.read_pickle("../figures/t_low_ood_all_per_metrics.pkl")
all_un_metrics_df = pd.read_pickle("../figures/t_low_ood_all_un_metrics.pkl")

unc_means = pd.concat(all_un_metrics_df).iloc[0::2, :]
unc_stds = pd.concat(all_un_metrics_df).iloc[1::2, :]
per_means = pd.concat(all_per_metrics_df).iloc[0::2, :]
per_stds = pd.concat(all_per_metrics_df).iloc[1::2, :]

unc = pd.merge(unc_means, unc_stds, on="non_target_frac")
per = pd.merge(per_means, per_stds, on="non_target_frac")
merged = pd.merge(unc, per, on="non_target_frac")

plt.rc("font", family="Helvetica Light", serif="Helvetica Light", size=20)
plt.rcParams["axes.linewidth"] = 1.5
plt.rcParams["xtick.major.size"] = 8
plt.rcParams["xtick.major.width"] = 1.5
plt.rcParams["ytick.major.size"] = 8
plt.rcParams["ytick.major.width"] = 1.5
plt.rcParams["ytick.direction"] = "in"
plt.rcParams["xtick.direction"] = "in"
plt.rcParams["legend.markerscale"] = 2
plt.rcParams["mathtext.it"] = "Helvetica Light:italic"
plt.rcParams["mathtext.rm"] = "Helvetica Light"
plt.rcParams["mathtext.default"] = "regular"

COLORS = ["#084c61", "#db504a"]
ECOLORS = ["#084c61a0", "#db4f4aa2"]
ELINEWIDTH = 1.5
LINEWIDTH = 2
MARKERSIZE = 10
CAPSIZE = 5
CAPSTICK = 2
MARKEREDGEWIDTH = 2

fig, axs = plt.subplots(2, 2, figsize=(18, 14), sharex=True, sharey=True)

plot_unc(
    axs[0, 0],
    axs[0, 0].twinx(),
    "Test_MAE",
    "Test_MAE",
    "MAE",
    show_legend=True,
    show_x_label=False,
    show_y_label=True,
)
plot_unc(
    axs[0, 1],
    axs[0, 1].twinx(),
    "Test_RMSE",
    "Test_RMSE",
    "RMSE",
    show_legend=False,
    show_x_label=False,
    show_y_label=False,
)
plot_unc(
    axs[1, 0],
    axs[1, 0].twinx(),
    "Test_MSE",
    "Test_MSE",
    "MSE",
    show_legend=False,
    show_x_label=True,
    show_y_label=True,
)
plot_unc(
    axs[1, 1],
    axs[1, 1].twinx(),
    "Test_R2",
    "Test_R2",
    r"$R^{2}$",
    show_legend=False,
    show_x_label=True,
    show_y_label=False,
)

plt.subplots_adjust(wspace=0.16, hspace=0.07)
plt.savefig("../figures/errorbar_t_low_uncertainty.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
def plot_unc(
    ax, ax2, y_metric, ylabel, show_legend=False, show_x_label=True, show_y_label=True
):

    ax.plot(
        merged["non_target_frac"],
        merged["Mean_Epistemic_Unc_x"],
        "o-",
        linewidth=LINEWIDTH,
        markersize=MARKERSIZE,
        markeredgewidth=MARKEREDGEWIDTH,
        color=COLORS[0],
        markerfacecolor=COLORS[0],
        markeredgecolor="white",
        label="Epistemic",
    )
    ax.plot(
        merged["non_target_frac"],
        merged["Mean_Aleatoric_Unc_x"],
        "s--",
        linewidth=LINEWIDTH,
        markersize=MARKERSIZE,
        markeredgewidth=MARKEREDGEWIDTH,
        color=COLORS[0],
        markerfacecolor=COLORS[0],
        markeredgecolor="white",
        label="Aleatoric",
    )
    ax.plot(
        merged["non_target_frac"],
        merged["Mean_Total_Unc_x"],
        "^-",
        linewidth=LINEWIDTH,
        markersize=MARKERSIZE,
        markeredgewidth=MARKEREDGEWIDTH,
        color=COLORS[0],
        markerfacecolor=COLORS[0],
        markeredgecolor="white",
        label="Total",
    )

    if show_legend:
        ax.legend(frameon=False, loc="upper right")

    if show_x_label:
        ax.set_xlabel("Low T Material (%)")
    if show_y_label:
        ax.set_ylabel("Uncertainty", color=COLORS[0])

    ax.tick_params(axis="y", labelcolor=COLORS[0])
    ax.set_xticks(np.arange(0.0, 1.1, 0.1))
    ax.set_xticklabels([int(x * 100) for x in np.arange(0.0, 1.1, 0.1)])

    ax2.plot(
        merged["non_target_frac"],
        merged[y_metric + "_x"],
        "s--",
        linewidth=LINEWIDTH,
        markersize=MARKERSIZE,
        markeredgewidth=MARKEREDGEWIDTH,
        color=COLORS[1],
        markerfacecolor=COLORS[1],
        markeredgecolor="white",
    )

    ax2.set_ylabel(ylabel, color=COLORS[1])
    ax2.tick_params(axis="y", labelcolor=COLORS[1])


unc_means = pd.concat(all_un_metrics_df).iloc[0::2, :]
unc_stds = pd.concat(all_un_metrics_df).iloc[1::2, :]
per_means = pd.concat(all_per_metrics_df).iloc[0::2, :]
per_stds = pd.concat(all_per_metrics_df).iloc[1::2, :]

unc = pd.merge(unc_means, unc_stds, on="non_target_frac")
per = pd.merge(per_means, per_stds, on="non_target_frac")
merged = pd.merge(unc, per, on="non_target_frac")

plt.rc("font", family="Helvetica Light", serif="Helvetica Light", size=20)
plt.rcParams["axes.linewidth"] = 1.5
plt.rcParams["xtick.major.size"] = 8
plt.rcParams["xtick.major.width"] = 1.5
plt.rcParams["ytick.major.size"] = 8
plt.rcParams["ytick.major.width"] = 1.5
plt.rcParams["ytick.direction"] = "in"
plt.rcParams["xtick.direction"] = "in"
plt.rcParams["legend.markerscale"] = 2
plt.rcParams["mathtext.it"] = "Helvetica Light:italic"
plt.rcParams["mathtext.rm"] = "Helvetica Light"
plt.rcParams["mathtext.default"] = "regular"

COLORS = ["#084c61", "#db504a"]
ECOLORS = ["#084c61a0", "#db4f4aa2"]
ELINEWIDTH = 1.5
LINEWIDTH = 2
MARKERSIZE = 10
CAPSIZE = 5
CAPSTICK = 2
MARKEREDGEWIDTH = 2

fig, axs = plt.subplots(2, 2, figsize=(18, 14), sharex=True, sharey=True)

plot_unc(
    axs[0, 0],
    axs[0, 0].twinx(),
    "Test_MAE",
    "MAE",
    show_legend=True,
    show_x_label=False,
    show_y_label=True,
)

plot_unc(
    axs[0, 1],
    axs[0, 1].twinx(),
    "Test_RMSE",
    "RMSE",
    show_legend=False,
    show_x_label=False,
    show_y_label=False,
)

plot_unc(
    axs[1, 0],
    axs[1, 0].twinx(),
    "Test_MSE",
    "MSE",
    show_legend=False,
    show_x_label=True,
    show_y_label=True,
)

plot_unc(
    axs[1, 1],
    axs[1, 1].twinx(),
    "Test_R2",
    r"$R^{2}$",
    show_legend=False,
    show_x_label=True,
    show_y_label=False,
)

plt.subplots_adjust(wspace=0.16, hspace=0.07)
plt.savefig("no_errorbar_t_low_uncertainty.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
all_results = []
root_dir = "./uncertainty_T_low_results/not_target_frac_1.0"
for split in sorted(os.listdir(root_dir)):
    split_path = os.path.join(root_dir, split)

    if os.path.isdir(split_path) and split.startswith("run_"):

        for model in sorted(os.listdir(split_path)):
            model_path = os.path.join(split_path, model)

            if os.path.isdir(model_path) and model.startswith("model_"):

                test_file = os.path.join(model_path, "results_enn_test_set.csv")

                if os.path.exists(test_file):
                    df = pd.read_csv(test_file)
                    all_results.append(df)

In [ ]:
results = pd.DataFrame(
    np.mean(all_results, axis=0),
    columns=["y_test", "y_pred", "aleatoric", "epistemic", "total"],
)

results.to_csv(
    "../figures/scatter_plot_residual_error_total_unc_low_t.csv", index=False
)

results["residual"] = abs(results["y_test"] - results["y_pred"])
results["nor_epist"] = results["epistemic"] / results["total"]
results["nor_aleat"] = results["aleatoric"] / results["total"]

In [ ]:
results.sort_values("y_test", inplace=True)

In [ ]:
plt.rc("font", family="Helvetica Light", serif="Helvetica Light", size=20)
plt.rcParams["axes.linewidth"] = 1.5
plt.rcParams["xtick.major.size"] = 8
plt.rcParams["xtick.major.width"] = 1.5
plt.rcParams["ytick.major.size"] = 8
plt.rcParams["ytick.major.width"] = 1.5
plt.rcParams["ytick.direction"] = "in"
plt.rcParams["xtick.direction"] = "in"
plt.rcParams["legend.markerscale"] = 2
plt.rcParams["mathtext.it"] = "Helvetica Light:italic"
plt.rcParams["mathtext.rm"] = "Helvetica Light"
plt.rcParams["mathtext.default"] = "regular"

plt.figure(figsize=(10, 8))
scatter = plt.scatter(
    results["total"],
    results["residual"],
    c=results["nor_epist"],
    cmap="winter",
    s=60,
    alpha=0.7,
    edgecolors="black",
    linewidths=1.4,
    vmin=0.0,
    vmax=1.0,
)

cbar = plt.colorbar(scatter)
cbar.set_label("Uncertainty", labelpad=-80)

cbar.set_ticks([0.0, 1.0])

cbar.set_ticklabels(["Aleatoric", "Epistemic"])

plt.ylabel("Total Uncertainty")
plt.xlabel("Residual Error")
plt.xlim(-0.2, 7)
plt.ylim(-0.1, 6)
plt.savefig("../figures/t_low_parity_plot.png", dpi=300, bbox_inches="tight")
plt.show()

# 7- Cleaning StarryDataset

In [ ]:
def create_strata(df, n_bins=5):
    """Creates a combined stratification column based on kL and T quantiles."""

    df[f"kL_bins"] = pd.qcut(df["kL"], q=n_bins, labels=False, duplicates="drop")
    df[f"T_bins"] = pd.qcut(df["T"], q=n_bins, labels=False, duplicates="drop")

    df["Strata"] = df["T_bins"].astype(str) + "_" + df["kL_bins"].astype(str)
    return df

In [ ]:
def run_experiment(
    train_set: pd.DataFrame,
    val_set: pd.DataFrame,
    test_set: pd.DataFrame,
    test_set_name: str,
    output_dir: Union[str, Path],
) -> None:
    """
    Run experiments across multiple random seeds and model instances.

    Parameters
    ----------
    train_set : pandas.DataFrame
        Train set.
    val_set: pandas.DataFrame
        Val set.
    test_set : pandas.DataFrame
        External test set.
    test_set_name : str
        Name of external test set.
    output_dir : str or pathlib.Path
        Directory path for saving outputs, model checkpoints, and split files.
    """
    device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")

    output_dir = Path(output_dir)
    run_dir = output_dir / f"run_1"
    run_dir.mkdir(parents=True, exist_ok=True)

    print("Train shape:", train_set.shape)

    X_val = val_set.iloc[:, 1:-2].to_numpy()
    X_train = train_set.iloc[:, 1:-2].to_numpy()
    X_test = test_set.iloc[:, 1:-2].to_numpy()
    y_val = val_set.iloc[:, -2].to_numpy()
    y_train = train_set.iloc[:, -2].to_numpy()
    y_test = test_set.iloc[:, -2].to_numpy()

    split_dict = {
        "X_train": X_train,
        "X_val": X_val,
        "X_test": X_test,
        "y_train": y_train,
        "y_val": y_val,
        "y_test": y_test,
    }

    with open(run_dir / "dataset_split.pkl", "wb") as f:
        pickle.dump(split_dict, f, protocol=pickle.HIGHEST_PROTOCOL)

    in_features = X_train.shape[1]
    print(f"Input features for model: {in_features}")

    train_loader, val_loader, test_loader = create_dataloaders(
        X_train,
        X_val,
        X_test,
        y_train,
        y_val,
        y_test,
        batch_size=CONFIG["batch_size"],
        shuffle_train=CONFIG["shuffle_train"],
    )

    dataloaders = {"train": train_loader, "val": val_loader, test_set_name: test_loader}

    for model_idx in range(1, 6):
        print(f"\n--- Training model {model_idx} on Run 1 ---")
        set_seed(42 + model_idx)

        model_dir = run_dir / f"model_{model_idx}"
        model_dir.mkdir(parents=True, exist_ok=True)
        best_model_path = model_dir / f"models/ffnn_model_best.pth"

        if best_model_path.exists():
            print("Best model found — skipping training.")

        else:

            model = FFNN(
                in_features,
                CONFIG["model_params"]["n_layers"],
                CONFIG["model_params"]["act_fn"],
                CONFIG["model_params"]["num_neu_list"],
                CONFIG["model_params"]["p"],
            )
            model.apply(lambda m: init_weights(m, nonlinearity=model.act_fn_name))
            model = model.to(device)

            optimizer = getattr(
                torch.optim, CONFIG["training_params"]["optimizer_name"]
            )(
                model.parameters(),
                lr=CONFIG["training_params"]["learning_rate"],
                weight_decay=CONFIG["training_params"]["weight_decay"],
            )
            scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
                optimizer,
                mode="min",
                factor=0.5,
                patience=CONFIG["training_params"]["scheduler_patience"],
            )

            early_stopper = EarlyStopper(
                patience=CONFIG["training_params"]["early_stopping_patience"],
                delta=CONFIG["training_params"]["early_stopping_delta"],
                minimize=True,
            )

            best_val_rmse, best_model_path = train_and_val_proc(
                model,
                optimizer,
                evidential_regression,
                CONFIG["training_params"]["epochs"],
                scheduler,
                dataloaders,
                early_stopper,
                device,
                model_dir,
                lamb_loss=CONFIG["training_params"]["loss_lamb"],
                target_inverse_transform=(
                    CONFIG["target_transform"]["inverse_function_for_metrics"]
                    if CONFIG["target_transform"]["enabled"]
                    else None
                ),
            )

            print(f"Model {model_idx} best validation RMSE: {best_val_rmse:.3f}")

        final_model = FFNN(
            in_features,
            CONFIG["model_params"]["n_layers"],
            CONFIG["model_params"]["act_fn"],
            CONFIG["model_params"]["num_neu_list"],
            CONFIG["model_params"]["p"],
        )
        final_model.load_state_dict(torch.load(best_model_path, map_location=device))
        final_model.to(device)

        test_proc(
            final_model,
            nig_nll,
            dataloaders,
            device,
            model_dir,
            target_inverse_transform=(
                CONFIG["target_transform"]["inverse_function_for_metrics"]
                if CONFIG["target_transform"]["enabled"]
                else None
            ),
            key=test_set_name,
        )

        val_proc(
            final_model,
            nig_nll,
            dataloaders,
            device,
            model_dir,
            target_inverse_transform=(
                CONFIG["target_transform"]["inverse_function_for_metrics"]
                if CONFIG["target_transform"]["enabled"]
                else None
            ),
        )

In [ ]:
total_fv_df = (
    pd.read_parquet("../feature_vectors/total_dataset.parquet")
    .sample(frac=1, random_state=42)
    .reset_index(drop=True)
)
starry_fv_df = pd.read_parquet("../feature_vectors/unique_starrydataset.parquet")

In [ ]:
total_fv_df = create_strata(total_fv_df)

df_train, df_val = train_test_split(
    total_fv_df,
    test_size=400,
    stratify=total_fv_df["Strata"],
    random_state=42,
)

train_set = (
    df_train.drop(columns=["T_bins", "kL_bins", "Strata"])
    .sample(frac=1, random_state=42)
    .reset_index(drop=True)
)

val_set = (
    df_val.drop(columns=["T_bins", "kL_bins", "Strata"])
    .sample(frac=1, random_state=42)
    .reset_index(drop=True)
)

In [ ]:
run_experiment(
    train_set, val_set, starry_fv_df, "starrydataset", "./starrydataset_screen"
)

In [ ]:
df_screen = pd.read_csv(
    f"./starrydataset_screen/run_1/model_1/results_enn_starrydataset_set.csv"
)

results_df = pd.concat([starry_fv_df.iloc[:, [0, -3]], df_screen], axis=1)

In [ ]:
random_precision = (results_df["y_test"] <= 1).sum() / len(results_df)

screened_epistemic = results_df[
    (results_df["y_pred"].between(0, 1))
    & (results_df["epistemic_unc"] <= results_df["epistemic_unc"].quantile(0.25))
]

screened_aleatoric = results_df[
    (results_df["y_pred"].between(0, 1))
    & (results_df["aleatoric_unc"] <= results_df["aleatoric_unc"].quantile(0.25))
]

screened_total = results_df[
    (results_df["y_pred"].between(0, 1))
    & (results_df["total_unc"] <= results_df["total_unc"].quantile(0.25))
]

baseline = results_df[results_df["y_pred"].between(0, 1)]

precision_epistemic = (screened_epistemic["y_test"] <= 1).sum() / len(
    screened_epistemic
)
precision_aleatoric = (screened_aleatoric["y_test"] <= 1).sum() / len(
    screened_aleatoric
)
precision_total = (screened_total["y_test"] <= 1).sum() / len(screened_total)
baseline_precision = (baseline["y_test"] <= 1).sum() / len(baseline)

print(f"Total compositions: {len(results_df)}")
print(f"True low kL compositions (y_test <= 1): {(results_df['y_test'] <= 1).sum()}")
print(f"Random screening precision: {random_precision:.3f}")
print()
print(f"Baseline compositions (without uncertainty filter): {len(baseline)}")
print(f"Baseline precision: {baseline_precision:.3f}")
print()
print(f"Screened compositions (epistemic filter): {len(screened_epistemic)}")
print(f"Precision with epistemic filter: {precision_epistemic:.3f}")
print(
    f"Improvement over baseline: {(precision_epistemic - baseline_precision):.3f} ({(precision_epistemic - baseline_precision) / baseline_precision * 100:.1f}%)"
)
print()
print(f"Screened compositions (aleatoric filter): {len(screened_aleatoric)}")
print(f"Precision with aleatoric filter: {precision_aleatoric:.3f}")
print(
    f"Improvement over baseline: {(precision_aleatoric - baseline_precision):.3f} ({(precision_aleatoric - baseline_precision) / baseline_precision * 100:.1f}%)"
)
print()
print(f"Screened compositions (total uncertainty filter): {len(screened_total)}")
print(f"Precision with total uncertainty filter: {precision_total:.3f}")
print(
    f"Improvement over baseline: {(precision_total - baseline_precision):.3f} ({(precision_total - baseline_precision) / baseline_precision * 100:.1f}%)"
)
print()
print(
    f"Improvement over random - baseline: {(baseline_precision - random_precision) / random_precision * 100:.1f}%"
)
print(
    f"Improvement over random - epistemic filter: {(precision_epistemic - random_precision) / random_precision * 100:.1f}%"
)

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score
import numpy as np

y_true_full = (results_df["y_test"] <= 1).astype(int)

# random — randomly select same number of compositions as baseline
np.random.seed(42)
y_pred_random = np.zeros(len(results_df), dtype=int)
random_indices = np.random.choice(len(results_df), size=len(baseline), replace=False)
y_pred_random[random_indices] = 1

y_pred_baseline = ((results_df["y_pred"].between(0, 1))).astype(int)

y_pred_epistemic = (
    (results_df["y_pred"].between(0, 1)) & (results_df["epistemic_unc"] <= 0.0792)
).astype(int)

print("Random screening:")
print(f"  Precision: {precision_score(y_true_full, y_pred_random):.3f}")
print(f"  Recall:    {recall_score(y_true_full, y_pred_random):.3f}")
print(f"  F1:        {f1_score(y_true_full, y_pred_random):.3f}")

print("\nBaseline (point prediction only):")
print(f"  Precision: {precision_score(y_true_full, y_pred_baseline):.3f}")
print(f"  Recall:    {recall_score(y_true_full, y_pred_baseline):.3f}")
print(f"  F1:        {f1_score(y_true_full, y_pred_baseline):.3f}")

print("\nENN with epistemic filter:")
print(f"  Precision: {precision_score(y_true_full, y_pred_epistemic):.3f}")
print(f"  Recall:    {recall_score(y_true_full, y_pred_epistemic):.3f}")
print(f"  F1:        {f1_score(y_true_full, y_pred_epistemic):.3f}")

print("\nEnrichment factor:")
print(f"  Baseline vs random:  {baseline_precision / random_precision:.2f}x")
print(f"  Epistemic vs random: {precision_epistemic / random_precision:.2f}x")

In [ ]:
plt.rc("font", family="Helvetica Light", serif="Helvetica Light", size=16)
plt.rcParams["axes.linewidth"] = 1.5
plt.rcParams["xtick.major.size"] = 4
plt.rcParams["xtick.major.width"] = 1.5
plt.rcParams["ytick.major.size"] = 4
plt.rcParams["ytick.major.width"] = 1.5
plt.rcParams["ytick.direction"] = "in"
plt.rcParams["xtick.direction"] = "in"
plt.rcParams["legend.markerscale"] = 2
plt.rcParams["mathtext.it"] = "Helvetica Light:italic"
plt.rcParams["mathtext.rm"] = "Helvetica Light"
plt.rcParams["mathtext.default"] = "regular"

fig, axes = plt.subplots(2, 2, figsize=(14, 12))

methods = ["Random", "ENN", "ENN +\nEpistemic Filter"]
COLORS = ["#26547c97", "#ef476e96", "#ffd1668b"]

precision_vals = [0.324, 0.516, 0.642]
recall_vals = [0.316, 0.504, 0.259]
f1_vals = [0.320, 0.510, 0.369]
enrichment_vals = [1.00, 1.61, 2.00]

ax = axes[0, 0]
bars = ax.bar(
    methods, precision_vals, color=COLORS, width=0.5, edgecolor="black", linewidth=1.5
)
ax.set_ylabel("Precision")
ax.set_ylim(0, 0.8)
ax.set_xticklabels([])

for bar, val in zip(bars, precision_vals):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.01,
        f"{val:.3f}",
        ha="center",
        va="bottom",
    )

ax = axes[0, 1]
bars = ax.bar(
    methods, recall_vals, color=COLORS, width=0.5, edgecolor="black", linewidth=1.5
)
ax.set_ylabel("Recall")
ax.set_ylim(0, 0.8)
ax.set_xticklabels([])

for bar, val in zip(bars, recall_vals):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.01,
        f"{val:.3f}",
        ha="center",
        va="bottom",
    )

ax = axes[1, 0]
bars = ax.bar(
    methods, f1_vals, color=COLORS, width=0.5, edgecolor="black", linewidth=1.5
)
ax.set_ylabel("F1 Score")
ax.set_ylim(0, 0.8)
ax.set_xticks(range(len(methods)))
ax.set_xticklabels(methods)


for bar, val in zip(bars, f1_vals):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.01,
        f"{val:.3f}",
        ha="center",
        va="bottom",
    )

ax = axes[1, 1]
bars = ax.bar(
    methods, enrichment_vals, color=COLORS, width=0.5, edgecolor="black", linewidth=1.5
)
ax.set_ylabel("Enrichment Factor (x)")
ax.set_ylim(0, 2.5)
ax.set_xticks(range(len(methods)))
ax.set_xticklabels(methods)

for bar, val in zip(bars, enrichment_vals):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + 0.01,
        f"{val:.2f}x",
        ha="center",
        va="bottom",
    )

plt.subplots_adjust(wspace=0.15, hspace=0.1)
plt.savefig("screening_comparison.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
dfs = []

for i in range(1, 6):

    df_screen = pd.read_csv(
        f"./starrydataset_screen/run_1/model_{i}/results_enn_starrydataset_set.csv"
    )

    dfs.append(df_screen)


combined_result_df = pd.concat(dfs)
mean_result_df = combined_result_df.groupby(level=0).mean(numeric_only=True)
std_result_df = combined_result_df.groupby(level=0).std(numeric_only=True)

mean_result_df = pd.concat([starry_fv_df.iloc[:, [0, -3]], mean_result_df], axis=1)
std_result_df = pd.concat([starry_fv_df.iloc[:, [0, -3]], std_result_df], axis=1)

In [ ]:
mean_result_df.to_csv(
    "./starrydataset_screen/run_1/starrydataset_mean.csv", index=False
)
std_result_df.to_csv("./starrydataset_screen/run_1/starrydataset_std.csv", index=False)

In [ ]:
mean_result_df.drop_duplicates(subset=["Formula"]).to_csv(
    "starrydataset_unique.csv", index=False
)

In [ ]:
mean_result_df["abs_err"] = (mean_result_df["y_test"] - mean_result_df["y_pred"]).abs()

# Rank each column independently (higher rank = higher value), then combine
mean_result_df["err_rank"] = mean_result_df["abs_err"].rank(ascending=False)
mean_result_df["unc_rank"] = mean_result_df["total_unc"].rank(ascending=False)
mean_result_df["combined_rank"] = (
    mean_result_df["err_rank"] + mean_result_df["unc_rank"]
)

mean_result_df_sorted = mean_result_df.sort_values("combined_rank")

In [ ]:
mean_result_df_sorted[mean_result_df_sorted["y_test"] < 0.1]